# Module 3 — MCP Architecture & Cross-LLM Evaluation
## Version B — Persistent Knowledge Base Build

**Purpose:** Build, shard, validate and persist the DuckDB BM25/FTS knowledge base.

**Repository note**
- This notebook preserves the executed experiment outputs for auditability and portfolio review.
- API credentials are loaded from environment variables or Google Colab secrets; no literal API keys are stored in this notebook.
- Large corpora and persistent BM25/DuckDB artifacts are intentionally excluded from Git and are accessed remotely or through the documented Kaggle artifact.
- This Build notebook contains no LLM benchmark execution; frozen prompt/MCP evaluation controls are owned by the runtime notebooks.

> Reproducibility dependencies and required secrets are documented in `README.md` and `requirements-module3.txt`.


```text
SECTION 0 — Environment
Cell 0 — Install Build Libraries
Cell 1 — API Keys + Imports + Global Build Config

SECTION 1 — Corpus Planning
Cell 2 — Corpus Registry + Inclusion Rules
Cell 3 — Source Validation + Ingestion Scope + Size Estimate
Cell 4 — Normalized Schema + Deduplication Policy

SECTION 2 — Build Persistent BM25 Index
Cell 5 — Download / Sync Active Corpora
Cell 6 — Discover Files + Validate Source Structures
Cell 7 — Normalize + Merge Corpora
Cell 8 — Deduplicate + Validate
Cell 9 — Telecom Tokenization Validation
Cell 10 — Build Persistent BM25 / FTS Shards
Cell 11 — Persist BM25 Artifact to Kaggle
Cell 12 — Database Health + Final Build Report

SECTION 3 — Build-Time Retrieval Validation + Optimization
Cell 13 — Baseline Global 21-Shard BM25 Retrieval
Cell 14 — Build-Time Optimized Retrieval + Source Groups
Cell 15 — Retrieval Quality + Latency Evaluation

SECTION 4 — Persistence Validation Utility
Cell 16 — Independent Kaggle Reload Validation


============================================================
BUILD NOTEBOOK — OBJECTIVE
============================================================

This notebook constructs Version B's persistent knowledge artifact:
**21 DuckDB BM25/FTS shards, 1,780,938 indexed records and ~69 GB**.

Cells 13–15 are historical **build-time retrieval experiments**. They
demonstrated the value of relevant-shard restriction, discriminative
query terms and controlled parallelism, but they are not the final
Benchmark v2 routing contract.

The later frozen runtime standardized routing with Version A:

query
→ deterministic 3GPP / TCC / Hybrid router
→ selected persistent BM25 shards

Search-profile groupings may remain as internal retrieval primitives,
but the public MCP tool exposes only `query` and `top_k`; the LLM does
not choose a profile, source family, collection or shard.

BUILD responsibility:
corpus acquisition → normalization → deduplication → tokenization
→ BM25/FTS construction → sharding → persistence → build diagnostics

RUNTIME responsibility:
restore artifact → deterministic router → retrieval/MCP validation
→ model orchestration → Benchmark v2

Do not interpret Cell 14's historical profile/top-3 experiment as the
final runtime contract. Any corpus/retrieval change after Benchmark v2
requires a new experimental version.
```


# **SECTION 0 — Environment**

## **Cell 0 — Install Build Libraries**

In [ ]:
# ============================================================
# CELL 0 — INSTALL BUILD DEPENDENCIES
# ============================================================

# Core build dependencies
!pip install -q \
    duckdb \
    huggingface_hub \
    pandas

# Kaggle CLI must be upgraded because older versions
# may not support the authentication / dataset workflow used here.
!pip install -q --upgrade kaggle


# ============================================================
# VERIFY KAGGLE INSTALLATION
# ============================================================

print("=" * 80)
print("BUILD DEPENDENCY CHECK")
print("=" * 80)

!kaggle --version

print("=" * 80)
print("✓ Version B build dependencies installed.")

## **Cell 1 — API Keys + Imports + Global Build Config**

In [ ]:
# ============================================================
# CELL 1 — API KEYS + IMPORTS + GLOBAL BUILD CONFIGURATION
# ============================================================

import os
import re
import time
import hashlib
import shutil

import duckdb
import pandas as pd

from pathlib import Path
from google.colab import userdata
from huggingface_hub import snapshot_download


# ============================================================
# OPTIONAL HUGGING FACE TOKEN
# ============================================================

try:
    HF_TOKEN = userdata.get(
        "HF_TOKEN"
    )

except Exception:
    HF_TOKEN = None


if HF_TOKEN:

    os.environ[
        "HF_TOKEN"
    ] = HF_TOKEN


# ============================================================
# BUILD WORKSPACE
# ============================================================

WORK_DIR = Path(
    "/tmp/telecom_bm25"
)

TCC_DIR = (
    WORK_DIR
    / "tcc"
)

GPP3_DIR = (
    WORK_DIR
    / "gsma_3gpp"
)

TEMP_DIR = (
    WORK_DIR
    / "duckdb_temp"
)

DB_PATH = (
    WORK_DIR
    / "telecom_corpus.duckdb"
)


WORK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TCC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GPP3_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# BUILD CONFIGURATION
# ============================================================

NUM_THREADS = (
    os.cpu_count()
    or 4
)

MEMORY_LIMIT = "32GB"

MIN_TEXT_LENGTH = 20

TOP_K_RESULTS = 5


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 80)
print("VERSION B — PERSISTENT BM25 KNOWLEDGE BASE BUILD")
print("=" * 80)

print(
    f"Work Directory      : "
    f"{WORK_DIR}"
)

print(
    f"Source Database     : "
    f"{DB_PATH}"
)

print(
    f"CPU Threads         : "
    f"{NUM_THREADS}"
)

print(
    f"DuckDB Memory Limit : "
    f"{MEMORY_LIMIT}"
)

print(
    f"Minimum Text Length : "
    f"{MIN_TEXT_LENGTH}"
)

print(
    f"Top-K Validation    : "
    f"{TOP_K_RESULTS}"
)

print(
    f"Hugging Face Access : "
    f"{'authenticated' if HF_TOKEN else 'public'}"
)

print("=" * 80)

print(
    "✓ Version B build environment configured."
)

VERSION B — PERSISTENT BM25 TELECOM KNOWLEDGE BASE
Work Directory       : /tmp/telecom_bm25
DuckDB Database      : /tmp/telecom_bm25/telecom_corpus.duckdb
CPU Threads          : 8
DuckDB Memory Limit  : 32GB
Top-K Results        : 5
HF Token             : loaded


# **SECTION 1 — Corpus Planning**

## **Cell 2 — Corpus Registry + Inclusion Rules**

In [ ]:
# ============================================================
# CELL 2 — CORPUS REGISTRY + INCLUSION RULES
# ============================================================

CORPUS_REGISTRY = {

    # --------------------------------------------------------
    # GSMA TELCO COMMON CORPUS
    # --------------------------------------------------------
    "tcc_full": {
        "source": "GSMA Telco Common Corpus",
        "source_family": "TCC",
        "dataset_repo": "GSMA/Telco-Common-Corpus",
        "ingest_pattern": "data/*.parquet",
        "enabled": True
    },

    # --------------------------------------------------------
    # DEDICATED 3GPP SPECIFICATION CORPUS
    # --------------------------------------------------------
    "gsma_3gpp": {
        "source": "GSMA 3GPP Standards Corpus",
        "source_family": "3GPP",
        "dataset_repo": "GSMA/3GPP",
        "ingest_pattern": "marked/**/raw.md",
        "enabled": True
    }
}


# ============================================================
# ACTIVE CORPUS VIEW
# ============================================================

ACTIVE_CORPORA = {
    name: config
    for name, config in CORPUS_REGISTRY.items()
    if config["enabled"]
}


# ============================================================
# VALIDATION
# ============================================================

print("=" * 80)
print("VERSION B — ACTIVE TELECOM CORPUS")
print("=" * 80)

for name, config in ACTIVE_CORPORA.items():

    print(
        f"{name:<15} | "
        f"{config['source_family']:<6} | "
        f"{config['dataset_repo']}"
    )

    print(
        f"{'':15} | "
        f"{'':6} | "
        f"Ingest: {config['ingest_pattern']}"
    )


print("=" * 80)

print(
    f"Active Corpus Sources : "
    f"{len(ACTIVE_CORPORA)}"
)

print("=" * 80)

TELECOM KNOWLEDGE CORPUS — PLANNED SOURCES
tcc_full             | TCC        | ACTIVE  | Priority 1
gsma_3gpp            | 3GPP       | ACTIVE  | Priority 1
gsma_docs            | GSMA       | PLANNED | Priority 2
tmforum_docs         | TMForum    | PLANNED | Priority 2
oran_docs            | O-RAN      | PLANNED | Priority 2
Active corpus sources : 2
  - tcc_full: GSMA/Telco-Common-Corpus [data/*.parquet]
  - gsma_3gpp: GSMA/3GPP [marked/**/raw.md]


## **Cell 3 — Source Validation + Ingestion Scope + Size Estimate**

In [ ]:
# ============================================================
# CELL 3 — VALIDATE SOURCES + INGESTION SCOPE + SIZE
# ============================================================

from huggingface_hub import HfApi
from fnmatch import fnmatch


api = HfApi(
    token=HF_TOKEN if "HF_TOKEN" in globals() else None
)


def matches_ingest_pattern(path: str, pattern: str) -> bool:
    """
    Checks whether a repository file matches the configured
    ingestion pattern.
    """

    if pattern is None:
        return False

    # Support our two current patterns explicitly
    if pattern == "data/*.parquet":
        return (
            path.startswith("data/")
            and path.lower().endswith(".parquet")
        )

    if pattern == "marked/**/raw.md":
        return (
            path.startswith("marked/")
            and path.lower().endswith("/raw.md")
        )

    return fnmatch(path, pattern)


def get_repo_summary(
    repo_id: str,
    ingest_pattern: str
) -> dict:
    """
    Validates a Hugging Face dataset repository and reports
    both repository structure and intended ingestion scope.
    """

    info = api.dataset_info(
        repo_id=repo_id,
        files_metadata=True
    )

    files = info.siblings or []

    repo_metadata_bytes = 0
    ingest_metadata_bytes = 0

    parquet_files = 0
    markdown_files = 0
    image_files = 0
    other_files = 0

    ingest_files = []

    for file in files:

        filename = file.rfilename
        size = getattr(file, "size", None) or 0

        repo_metadata_bytes += size

        lower_name = filename.lower()

        if lower_name.endswith(".parquet"):
            parquet_files += 1

        elif lower_name.endswith((".md", ".markdown")):
            markdown_files += 1

        elif lower_name.endswith(
            (".jpg", ".jpeg", ".png", ".gif", ".webp")
        ):
            image_files += 1

        else:
            other_files += 1

        if matches_ingest_pattern(
            filename,
            ingest_pattern
        ):
            ingest_files.append(filename)
            ingest_metadata_bytes += size

    return {
        "repo_id": repo_id,
        "available": True,

        # Repository-level structure
        "repo_file_count": len(files),
        "repo_metadata_size_gb":
            repo_metadata_bytes / (1024 ** 3),

        "parquet_files": parquet_files,
        "markdown_files": markdown_files,
        "image_files": image_files,
        "other_files": other_files,

        # Actual ingestion scope
        "ingest_pattern": ingest_pattern,
        "ingest_file_count": len(ingest_files),
        "ingest_metadata_size_gb":
            ingest_metadata_bytes / (1024 ** 3),

        "ingest_examples": ingest_files[:10]
    }


# ============================================================
# VALIDATE ACTIVE CORPORA
# ============================================================

SOURCE_SUMMARIES = {}

print("=" * 85)
print("ACTIVE CORPUS SOURCE VALIDATION")
print("=" * 85)

for corpus_name, config in ACTIVE_CORPORA.items():

    repo_id = config["dataset_repo"]
    ingest_pattern = config["ingest_pattern"]

    print(f"\nChecking       : {corpus_name}")
    print(f"Repository     : {repo_id}")
    print(f"Ingest Pattern : {ingest_pattern}")

    try:

        summary = get_repo_summary(
            repo_id,
            ingest_pattern
        )

        SOURCE_SUMMARIES[corpus_name] = summary

        print("Status         : AVAILABLE")

        print("\nRepository Structure")
        print(
            f"  Files        : "
            f"{summary['repo_file_count']:,}"
        )
        print(
            f"  Metadata Size: "
            f"{summary['repo_metadata_size_gb']:.2f} GB"
        )
        print(
            f"  Parquet      : "
            f"{summary['parquet_files']:,}"
        )
        print(
            f"  Markdown     : "
            f"{summary['markdown_files']:,}"
        )
        print(
            f"  Images       : "
            f"{summary['image_files']:,}"
        )
        print(
            f"  Other        : "
            f"{summary['other_files']:,}"
        )

        print("\nSelected Ingestion Scope")
        print(
            f"  Files        : "
            f"{summary['ingest_file_count']:,}"
        )
        print(
            f"  Metadata Size: "
            f"{summary['ingest_metadata_size_gb']:.2f} GB"
        )

        print("\nSelected File Examples")

        for path in summary["ingest_examples"]:
            print(f"  - {path}")

    except Exception as exc:

        SOURCE_SUMMARIES[corpus_name] = {
            "repo_id": repo_id,
            "available": False,
            "error": str(exc)
        }

        print("Status         : ERROR")
        print(f"Reason         : {exc}")


# ============================================================
# COMBINED INGESTION SUMMARY
# ============================================================

available_sources = [
    item
    for item in SOURCE_SUMMARIES.values()
    if item.get("available")
]

total_selected_files = sum(
    item["ingest_file_count"]
    for item in available_sources
)

total_selected_metadata_gb = sum(
    item["ingest_metadata_size_gb"]
    for item in available_sources
)

print("\n" + "=" * 85)
print("PLANNED INGESTION SUMMARY")
print("=" * 85)

print(
    f"Active sources             : "
    f"{len(ACTIVE_CORPORA)}"
)

print(
    f"Available sources          : "
    f"{len(available_sources)}"
)

print(
    f"Selected files             : "
    f"{total_selected_files:,}"
)

print(
    f"Selected metadata size     : "
    f"{total_selected_metadata_gb:.2f} GB"
)

print("=" * 85)

print(
    "\nNOTE: Hugging Face file metadata size may not equal the "
    "repository's displayed physical/Xet storage size."
)

print(
    "Capacity planning will use actual downloaded size during "
    "the ingestion stage."
)

ACTIVE CORPUS SOURCE VALIDATION

Checking       : tcc_full
Repository     : GSMA/Telco-Common-Corpus
Ingest Pattern : data/*.parquet
Status         : AVAILABLE

Repository Structure
  Files        : 102
  Metadata Size: 8.84 GB
  Parquet      : 100
  Markdown     : 1
  Images       : 0
  Other        : 1

Selected Ingestion Scope
  Files        : 100
  Metadata Size: 8.84 GB

Selected File Examples
  - data/tcc_001.parquet
  - data/tcc_002.parquet
  - data/tcc_003.parquet
  - data/tcc_004.parquet
  - data/tcc_005.parquet
  - data/tcc_006.parquet
  - data/tcc_007.parquet
  - data/tcc_008.parquet
  - data/tcc_009.parquet
  - data/tcc_010.parquet

Checking       : gsma_3gpp
Repository     : GSMA/3GPP
Ingest Pattern : marked/**/raw.md
Status         : AVAILABLE

Repository Structure
  Files        : 84,220
  Metadata Size: 5.21 GB
  Parquet      : 0
  Markdown     : 5,134
  Images       : 79,081
  Other        : 5

Selected Ingestion Scope
  Files        : 5,132
  Metadata Size: 0.81 GB

S

## **Cell 4 — Normalized Schema + Deduplication Policy**

In [ ]:
# ============================================================
# CELL 4 — NORMALIZED SCHEMA + DEDUPLICATION POLICY
# ============================================================


# ============================================================
# NORMALIZED DOCUMENT SCHEMA
# ============================================================

NORMALIZED_SCHEMA = [
    "doc_id",
    "source",
    "source_family",
    "collection",
    "identifier",
    "title",
    "release",
    "version",
    "date",
    "creator",
    "document_type",
    "source_path",
    "text",
    "text_hash"
]


# ============================================================
# DEDUPLICATION POLICY
# ============================================================

DEDUPLICATION_POLICY = {
    "method":
        "exact normalized-text hash",

    "hash_algorithm":
        "sha256",

    "cross_source_policy":
        "remove only exact-text duplicates",

    "preserve_source_metadata":
        True,

    "min_text_length":
        MIN_TEXT_LENGTH
}


# ============================================================
# DISPLAY PLANNING CONTRACT
# ============================================================

print("=" * 80)
print("NORMALIZED TELECOM CORPUS SCHEMA")
print("=" * 80)

for field in NORMALIZED_SCHEMA:
    print(
        f"  - {field}"
    )


print("\n" + "=" * 80)
print("DEDUPLICATION POLICY")
print("=" * 80)

print(
    f"Method                  : "
    f"{DEDUPLICATION_POLICY['method']}"
)

print(
    f"Hash Algorithm          : "
    f"{DEDUPLICATION_POLICY['hash_algorithm']}"
)

print(
    f"Cross-Source Policy     : "
    f"{DEDUPLICATION_POLICY['cross_source_policy']}"
)

print(
    f"Preserve Metadata       : "
    f"{DEDUPLICATION_POLICY['preserve_source_metadata']}"
)

print(
    f"Minimum Text Length     : "
    f"{DEDUPLICATION_POLICY['min_text_length']}"
)

print("=" * 80)
print("✓ Corpus normalization contract defined.")

NORMALIZED TELECOM CORPUS SCHEMA
  - doc_id
  - source
  - source_family
  - collection
  - identifier
  - title
  - release
  - version
  - date
  - creator
  - document_type
  - source_path
  - text
  - text_hash

ACTIVE SOURCE NORMALIZATION RULES

tcc_full
  Source Family : TCC
  Document Type : tcc_document

gsma_3gpp
  Source Family : 3GPP
  Document Type : 3gpp_specification

DEDUPLICATION POLICY
Exact text hash          : True
Cross-source exact only  : True
Minimum text length      : 20
Hash algorithm           : sha256


# **SECTION 2 — Build Persistent BM25 Index**

## **Cell 5 — Download / Sync Active Corpora**

In [ ]:
# ============================================================
# CELL 5 — DOWNLOAD / SYNC ACTIVE CORPORA
# ============================================================


# ============================================================
# LOCAL CORPUS DIRECTORIES
# ============================================================

CORPUS_DIRS = {
    "tcc_full": TCC_DIR,
    "gsma_3gpp": GPP3_DIR
}


# ============================================================
# HELPERS
# ============================================================

def get_directory_size(path: Path) -> int:
    """Return total directory size in bytes."""

    if not path.exists():
        return 0

    return sum(
        file.stat().st_size
        for file in path.rglob("*")
        if file.is_file()
    )


def format_gb(size_bytes: int) -> float:
    """Convert bytes to GiB."""

    return (
        size_bytes
        / (1024 ** 3)
    )


def sync_corpus(
    corpus_name: str,
    config: dict
) -> dict:
    """
    Download only the configured ingestion scope for one corpus.

    Existing Hugging Face downloads are reused where possible.
    """

    if corpus_name not in CORPUS_DIRS:

        raise ValueError(
            f"No local directory configured "
            f"for corpus: {corpus_name}"
        )


    repo_id = (
        config["dataset_repo"]
    )

    ingest_pattern = (
        config["ingest_pattern"]
    )

    local_dir = (
        CORPUS_DIRS[corpus_name]
    )


    local_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    print("=" * 80)
    print(
        f"SYNCING CORPUS : "
        f"{corpus_name}"
    )
    print("=" * 80)

    print(
        f"Repository     : "
        f"{repo_id}"
    )

    print(
        f"Ingest Pattern : "
        f"{ingest_pattern}"
    )

    print(
        f"Local Path     : "
        f"{local_dir}"
    )


    start = time.perf_counter()


    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=str(local_dir),
        allow_patterns=[
            ingest_pattern
        ],
        token=HF_TOKEN
    )


    elapsed = (
        time.perf_counter()
        - start
    )


    local_size = (
        get_directory_size(
            local_dir
        )
    )


    return {
        "corpus_name":
            corpus_name,

        "repo_id":
            repo_id,

        "local_dir":
            str(local_dir),

        "size_bytes":
            local_size,

        "size_gb":
            format_gb(
                local_size
            ),

        "elapsed_seconds":
            elapsed
    }


# ============================================================
# SYNC ALL ACTIVE CORPORA
# ============================================================

CORPUS_DOWNLOADS = {}

total_start = (
    time.perf_counter()
)


for corpus_name, config in ACTIVE_CORPORA.items():

    result = sync_corpus(
        corpus_name,
        config
    )

    CORPUS_DOWNLOADS[
        corpus_name
    ] = result


total_elapsed = (
    time.perf_counter()
    - total_start
)


# ============================================================
# VALIDATE ACTUAL INGESTION FILES
# ============================================================

tcc_parquet_files = sorted(
    TCC_DIR.glob(
        "data/*.parquet"
    )
)

gpp3_markdown_files = sorted(
    GPP3_DIR.glob(
        "marked/**/raw.md"
    )
)


if not tcc_parquet_files:

    raise RuntimeError(
        "No TCC Parquet files were downloaded."
    )


if not gpp3_markdown_files:

    raise RuntimeError(
        "No GSMA/3GPP raw.md files were downloaded."
    )


# ============================================================
# DOWNLOAD SUMMARY
# ============================================================

total_bytes = sum(
    item["size_bytes"]
    for item
    in CORPUS_DOWNLOADS.values()
)


print("\n" + "=" * 80)
print("CORPUS DOWNLOAD SUMMARY")
print("=" * 80)


for name, result in CORPUS_DOWNLOADS.items():

    print(
        f"{name:<15} | "
        f"{result['size_gb']:>6.2f} GB | "
        f"{result['elapsed_seconds']:>8.2f} sec"
    )


print("-" * 80)

print(
    f"TCC Parquet Files     : "
    f"{len(tcc_parquet_files):,}"
)

print(
    f"3GPP raw.md Files     : "
    f"{len(gpp3_markdown_files):,}"
)

print(
    f"Downloaded Disk Size  : "
    f"{format_gb(total_bytes):.2f} GB"
)

print(
    f"Total Sync Time       : "
    f"{total_elapsed:.2f} sec"
)


# ============================================================
# SAMPLE INGESTION FILES
# ============================================================

print("\nExample TCC Files:")

for path in tcc_parquet_files[:5]:

    print(
        f"  - "
        f"{path.relative_to(TCC_DIR)}"
    )


print("\nExample 3GPP Files:")

for path in gpp3_markdown_files[:5]:

    print(
        f"  - "
        f"{path.relative_to(GPP3_DIR)}"
    )


print("=" * 80)

print(
    "✓ Active telecom corpora downloaded "
    "and ingestion files validated."
)

SYNCING CORPUS : tcc_full
Repository     : GSMA/Telco-Common-Corpus
Ingest Pattern : data/*.parquet
Local Path     : /tmp/telecom_bm25/tcc


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 100 files:   0%|          | 0/100 [00:00<?, ?it/s]

Files Downloaded : 303
Actual Disk Size : 8.84 GB
Elapsed Time     : 45.89 sec
SYNCING CORPUS : gsma_3gpp
Repository     : GSMA/3GPP
Ingest Pattern : marked/**/raw.md
Local Path     : /tmp/telecom_bm25/gsma_3gpp


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15052 files:   0%|          | 0/15052 [00:00<?, ?it/s]

Files Downloaded : 45,159
Actual Disk Size : 3.29 GB
Elapsed Time     : 3480.39 sec

CORPUS DOWNLOAD SUMMARY
tcc_full        |    303 files |   8.84 GB
gsma_3gpp       | 45,159 files |   3.29 GB
--------------------------------------------------------------------------------
Total Files      : 45,462
Actual Disk Size : 12.12 GB
Total Sync Time  : 3529.49 sec


## **Cell 6 — Discover Files + Inspect Schemas**

In [ ]:
# ============================================================
# CELL 6 — DISCOVER FILES + VALIDATE SOURCE STRUCTURES
# ============================================================


# ============================================================
# VERIFY INGESTION FILES
# ============================================================

if not tcc_parquet_files:

    raise RuntimeError(
        "No TCC Parquet files were found."
    )


if not gpp3_markdown_files:

    raise RuntimeError(
        "No GSMA/3GPP raw.md files were found."
    )


print("=" * 80)
print("SOURCE STRUCTURE VALIDATION")
print("=" * 80)

print(
    f"TCC Parquet Files : "
    f"{len(tcc_parquet_files):,}"
)

print(
    f"3GPP raw.md Files : "
    f"{len(gpp3_markdown_files):,}"
)


# ============================================================
# TCC PARQUET SCHEMA
# ============================================================

tcc_glob = str(
    TCC_DIR
    / "data"
    / "*.parquet"
).replace(
    "\\",
    "/"
).replace(
    "'",
    "''"
)


conn = duckdb.connect()


tcc_schema_rows = conn.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{tcc_glob}',
        union_by_name = true
    )
    """
).fetchall()


TCC_SCHEMA = {
    row[0]: row[1]
    for row in tcc_schema_rows
}


# ============================================================
# REQUIRED TCC INGESTION FIELDS
# ============================================================

REQUIRED_TCC_COLUMNS = {
    "identifier",
    "collection",
    "title",
    "date",
    "creator",
    "text"
}


missing_tcc_columns = (
    REQUIRED_TCC_COLUMNS
    - set(TCC_SCHEMA)
)


if missing_tcc_columns:

    conn.close()

    raise RuntimeError(
        "TCC schema is missing required "
        "ingestion columns: "
        f"{sorted(missing_tcc_columns)}"
    )


print("\n" + "=" * 80)
print("TCC PARQUET SCHEMA")
print("=" * 80)


for column, dtype in TCC_SCHEMA.items():

    print(
        f"{column:<20} : "
        f"{dtype}"
    )


print(
    "\n✓ Required TCC ingestion "
    "columns are available."
)


# ============================================================
# TCC SAMPLE RECORD
# ============================================================

tcc_sample = conn.execute(
    f"""
    SELECT
        identifier,
        collection,
        title,
        date,
        creator,
        LEFT(text, 500) AS text_preview

    FROM read_parquet(
        '{tcc_glob}',
        union_by_name = true
    )

    LIMIT 1
    """
).df()


conn.close()


print("\n" + "=" * 80)
print("TCC SAMPLE RECORD")
print("=" * 80)

display(
    tcc_sample
)


# ============================================================
# CANONICAL 3GPP PATH METADATA PARSER
# ============================================================

def parse_3gpp_path(
    path: Path
) -> dict:
    """
    Extract release, series and specification identifier
    from a GSMA/3GPP repository path.

    Examples:
        23501   -> 23.501
        32111-1 -> 32.111-1
    """

    relative_path = (
        path.relative_to(
            GPP3_DIR
        )
    )

    parts = (
        relative_path.parts
    )


    release = None
    series = None
    spec_number = None


    for part in parts:

        if re.fullmatch(
            r"Rel-\d+",
            part
        ):

            release = part


        elif re.fullmatch(
            r"\d+_series",
            part
        ):

            series = (
                part.replace(
                    "_series",
                    ""
                )
            )


    parent = (
        path.parent.name
    )


    if re.fullmatch(
        r"\d+(?:-\d+)*",
        parent
    ):

        spec_number = parent


    identifier = None


    if spec_number:

        match = re.fullmatch(
            r"(\d{2})(\d{3})(.*)",
            spec_number
        )


        if match:

            identifier = (
                f"{match.group(1)}."
                f"{match.group(2)}"
                f"{match.group(3)}"
            )


    return {
        "release":
            release,

        "series":
            series,

        "spec_number":
            spec_number,

        "identifier":
            identifier,

        "source_path":
            str(relative_path)
    }


# ============================================================
# VALIDATE 3GPP PATH STRUCTURE
# ============================================================

parsed_count = 0

unparsed_examples = []


for path in gpp3_markdown_files:

    metadata = (
        parse_3gpp_path(
            path
        )
    )


    if (
        metadata["release"]
        and metadata["series"]
        and metadata["identifier"]
    ):

        parsed_count += 1


    elif len(
        unparsed_examples
    ) < 10:

        unparsed_examples.append(
            metadata["source_path"]
        )


unparsed_count = (
    len(gpp3_markdown_files)
    - parsed_count
)


print("\n" + "=" * 80)
print("3GPP PATH METADATA VALIDATION")
print("=" * 80)

print(
    f"Successfully Parsed : "
    f"{parsed_count:,} / "
    f"{len(gpp3_markdown_files):,}"
)

print(
    f"Unparsed            : "
    f"{unparsed_count:,}"
)


if unparsed_examples:

    print(
        "\nUnparsed Examples:"
    )

    for path in unparsed_examples:

        print(
            f"  - {path}"
        )


# ============================================================
# SAMPLE 3GPP PATH METADATA
# ============================================================

print("\nSample 3GPP Specifications:")


for path in gpp3_markdown_files[:5]:

    metadata = (
        parse_3gpp_path(
            path
        )
    )

    print(
        f"  {metadata['identifier']:<12} | "
        f"{metadata['release']:<8} | "
        f"{metadata['source_path']}"
    )


print("=" * 80)

print(
    "✓ Source structures validated "
    "for normalization."
)

SOURCE DISCOVERY
TCC Parquet Files : 100
3GPP raw.md Files : 15,052

TCC PARQUET SCHEMA
identifier           : VARCHAR
collection           : VARCHAR
open_type            : VARCHAR
curator              : VARCHAR
license              : VARCHAR
date                 : VARCHAR
title                : VARCHAR
creator              : VARCHAR
language             : VARCHAR
language_type        : VARCHAR
word_count           : INTEGER
token_count          : INTEGER
text                 : VARCHAR

TCC SAMPLE RECORD
identifier           : RFC8202
collection           : IETF-RFCs
open_type            : Open Government
curator              : Pleias/GSMA
license              : IETF Trust §4.c
date                 : 2017
title                : June 2017
creator              : L. Ginsberg, S. Previdi, W. Henderickx
language             : English
language_type        : Written
word_count           : 383
token_count          : 710
text                 : ### 9.2 Informative References

[Err4519]  RFC Erra

## **Cell 7 — Normalize + Merge Corpora**

In [ ]:
# ============================================================
# CELL 7 — NORMALIZE + MERGE CORPORA INTO DUCKDB
# ============================================================


# ============================================================
# NORMALIZATION HELPERS
# ============================================================

def normalize_text_for_hash(text: str) -> str:
    """
    Normalize whitespace for stable exact-text hashing.
    """

    return " ".join(
        str(text).split()
    )


def sha256_text(text: str) -> str:
    """
    Return SHA-256 hash of normalized document text.
    """

    return hashlib.sha256(
        normalize_text_for_hash(
            text
        ).encode(
            "utf-8",
            errors="ignore"
        )
    ).hexdigest()


def extract_3gpp_metadata(
    text: str,
    identifier: str
) -> dict:
    """
    Extract specification type, version and title
    from a GSMA/3GPP Markdown document.
    """

    header = text[:12000]


    # --------------------------------------------------------
    # SPECIFICATION TYPE + VERSION
    # --------------------------------------------------------

    version_match = re.search(
        r"3GPP\s+(TS|TR)\s+"
        r"(\d{2}\.\d{3}(?:-\d+)?)\s+"
        r"V([\d.]+)",
        header,
        flags=re.IGNORECASE
    )


    if version_match:

        spec_type = (
            version_match
            .group(1)
            .upper()
        )

        version = (
            version_match
            .group(3)
        )

    else:

        spec_type = "3GPP"
        version = None


    # --------------------------------------------------------
    # DOCUMENT TITLE
    # --------------------------------------------------------

    title = None


    for line in text.splitlines()[:100]:

        clean = re.sub(
            r"^#+\s*",
            "",
            line
        ).strip()

        clean = re.sub(
            r"[*_~]+",
            "",
            clean
        ).strip()


        if (
            len(clean) >= 20
            and not clean.startswith("![")
            and "3gpp logo"
                not in clean.lower()
        ):

            title = clean[:500]
            break


    if not title:

        title = (
            f"3GPP "
            f"{spec_type} "
            f"{identifier}"
        )


    return {
        "version":
            version,

        "title":
            title,

        "spec_type":
            spec_type
    }


# ============================================================
# SOURCE CONFIGURATION
# ============================================================

TCC_CONFIG = (
    ACTIVE_CORPORA[
        "tcc_full"
    ]
)

GPP3_CONFIG = (
    ACTIVE_CORPORA[
        "gsma_3gpp"
    ]
)


# ============================================================
# INITIALIZE DUCKDB
# ============================================================

print("=" * 80)
print("BUILDING NORMALIZED TELECOM CORPUS")
print("=" * 80)


start_total = (
    time.perf_counter()
)


conn = duckdb.connect(
    str(DB_PATH)
)


conn.execute(
    f"SET threads TO "
    f"{NUM_THREADS}"
)

conn.execute(
    f"SET memory_limit = "
    f"'{MEMORY_LIMIT}'"
)

conn.execute(
    f"SET temp_directory = "
    f"'{str(TEMP_DIR)}'"
)

conn.execute(
    "SET preserve_insertion_order = false"
)


# ============================================================
# CREATE NORMALIZED CORPUS TABLE
# ============================================================

conn.execute("""
    DROP TABLE IF EXISTS telecom_corpus
""")


conn.execute("""
    CREATE TABLE telecom_corpus (

        doc_id VARCHAR,

        source VARCHAR,
        source_family VARCHAR,
        collection VARCHAR,

        identifier VARCHAR,
        title VARCHAR,

        release VARCHAR,
        version VARCHAR,
        date VARCHAR,
        creator VARCHAR,

        document_type VARCHAR,
        source_path VARCHAR,

        text VARCHAR,
        text_hash VARCHAR
    )
""")


# ============================================================
# VALIDATE NORMALIZED TABLE SCHEMA
# ============================================================

actual_schema = [
    row[0]
    for row in conn.execute(
        """
        DESCRIBE telecom_corpus
        """
    ).fetchall()
]


if actual_schema != NORMALIZED_SCHEMA:

    conn.close()

    raise RuntimeError(
        "Normalized DuckDB schema does not "
        "match the Cell 4 schema contract."
    )


print(
    "✓ Normalized DuckDB schema validated."
)


# ============================================================
# 1. INGEST TCC
# ============================================================

print(
    "\n[1/2] "
    "Normalizing full TCC corpus..."
)


tcc_start = (
    time.perf_counter()
)


conn.execute(
    f"""
    INSERT INTO telecom_corpus

    SELECT

        sha256(
            'TCC|'
            || COALESCE(
                identifier,
                ''
            )
            || '|'
            || filename
        ) AS doc_id,

        '{TCC_CONFIG["dataset_repo"]}'
            AS source,

        '{TCC_CONFIG["source_family"]}'
            AS source_family,

        collection,

        identifier,
        title,

        NULL AS release,
        NULL AS version,

        date,
        creator,

        CASE

            WHEN collection = '3GPP-TSG'
                THEN '3gpp_tsg_document'

            ELSE 'tcc_document'

        END AS document_type,

        filename AS source_path,

        text,

        sha256(
            regexp_replace(
                trim(text),
                '\\\\s+',
                ' ',
                'g'
            )
        ) AS text_hash

    FROM read_parquet(
        '{tcc_glob}',
        union_by_name = true,
        filename = true
    )

    WHERE text IS NOT NULL

      AND length(
          trim(text)
      ) >= {MIN_TEXT_LENGTH}
    """
)


tcc_count = conn.execute(
    """
    SELECT COUNT(*)

    FROM telecom_corpus

    WHERE source_family = 'TCC'
    """
).fetchone()[0]


tcc_elapsed = (
    time.perf_counter()
    - tcc_start
)


print(
    f"     ✓ "
    f"{tcc_count:,} "
    f"TCC records loaded"
    f" | "
    f"{tcc_elapsed:.2f}s"
)


# ============================================================
# 2. INGEST GSMA/3GPP MARKDOWN
# ============================================================

print(
    "\n[2/2] "
    "Normalizing GSMA/3GPP specifications..."
)


gpp_start = (
    time.perf_counter()
)


BATCH_SIZE = 250

batch = []

gpp_loaded = 0
gpp_skipped = 0


# ============================================================
# 3GPP BATCH INSERT
# ============================================================

def flush_3gpp_batch(records):
    """
    Insert one normalized 3GPP batch using the
    frozen normalized schema order.
    """

    if not records:
        return


    df_batch = pd.DataFrame(
        records,
        columns=NORMALIZED_SCHEMA
    )


    conn.register(
        "gpp_batch",
        df_batch
    )


    column_list = ", ".join(
        NORMALIZED_SCHEMA
    )


    conn.execute(
        f"""
        INSERT INTO telecom_corpus (
            {column_list}
        )

        SELECT
            {column_list}

        FROM gpp_batch
        """
    )


    conn.unregister(
        "gpp_batch"
    )


# ============================================================
# NORMALIZE 3GPP FILES
# ============================================================

for path in gpp3_markdown_files:

    path_meta = (
        parse_3gpp_path(
            path
        )
    )


    if not path_meta[
        "identifier"
    ]:

        gpp_skipped += 1
        continue


    text = path.read_text(
        encoding="utf-8",
        errors="ignore"
    )


    if (
        len(
            text.strip()
        )
        < MIN_TEXT_LENGTH
    ):

        gpp_skipped += 1
        continue


    content_meta = (
        extract_3gpp_metadata(
            text,
            path_meta[
                "identifier"
            ]
        )
    )


    text_hash = (
        sha256_text(
            text
        )
    )


    doc_id = hashlib.sha256(
        (
            f"{GPP3_CONFIG['dataset_repo']}|"
            + path_meta[
                "source_path"
            ]
        ).encode(
            "utf-8"
        )
    ).hexdigest()


    record = {

        "doc_id":
            doc_id,

        "source":
            GPP3_CONFIG[
                "dataset_repo"
            ],

        "source_family":
            GPP3_CONFIG[
                "source_family"
            ],

        "collection":
            "3GPP-Specifications",

        "identifier":
            path_meta[
                "identifier"
            ],

        "title":
            content_meta[
                "title"
            ],

        "release":
            path_meta[
                "release"
            ],

        "version":
            content_meta[
                "version"
            ],

        "date":
            None,

        "creator":
            "3GPP",

        "document_type":
            "3gpp_specification",

        "source_path":
            path_meta[
                "source_path"
            ],

        "text":
            text,

        "text_hash":
            text_hash
    }


    batch.append(
        record
    )


    if len(batch) >= BATCH_SIZE:

        flush_3gpp_batch(
            batch
        )

        gpp_loaded += (
            len(batch)
        )

        batch = []


        if (
            gpp_loaded
            % 2000
            < BATCH_SIZE
        ):

            print(
                f"     Loaded "
                f"{gpp_loaded:,} / "
                f"{len(gpp3_markdown_files):,}"
            )


# ============================================================
# FINAL 3GPP BATCH
# ============================================================

if batch:

    flush_3gpp_batch(
        batch
    )

    gpp_loaded += (
        len(batch)
    )


gpp_elapsed = (
    time.perf_counter()
    - gpp_start
)


# ============================================================
# CHECKPOINT
# ============================================================

conn.execute(
    "CHECKPOINT"
)


# ============================================================
# POST-MERGE VALIDATION
# ============================================================

total_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM telecom_corpus
    """
).fetchone()[0]


gpp_count = conn.execute(
    """
    SELECT COUNT(*)

    FROM telecom_corpus

    WHERE source_family = '3GPP'
    """
).fetchone()[0]


# ------------------------------------------------------------
# COUNT CONSISTENCY
# ------------------------------------------------------------

expected_total = (
    tcc_count
    + gpp_count
)


if total_count != expected_total:

    conn.close()

    raise RuntimeError(
        "Merged corpus count validation failed: "
        f"{total_count:,} != "
        f"{expected_total:,}"
    )


if gpp_count != gpp_loaded:

    conn.close()

    raise RuntimeError(
        "3GPP inserted-record count does not "
        "match the batch tracker: "
        f"{gpp_count:,} != "
        f"{gpp_loaded:,}"
    )


if (
    gpp_loaded
    + gpp_skipped
    != len(
        gpp3_markdown_files
    )
):

    conn.close()

    raise RuntimeError(
        "3GPP file accounting validation failed."
    )


# ============================================================
# SOURCE SUMMARY
# ============================================================

source_summary = conn.execute(
    """
    SELECT
        source_family,
        document_type,
        COUNT(*) AS records

    FROM telecom_corpus

    GROUP BY
        source_family,
        document_type

    ORDER BY
        source_family,
        document_type
    """
).df()


total_elapsed = (
    time.perf_counter()
    - start_total
)


# ============================================================
# REPORT
# ============================================================

print("\n" + "=" * 80)
print("NORMALIZED CORPUS SUMMARY")
print("=" * 80)


print(
    f"TCC Records          : "
    f"{tcc_count:,}"
)

print(
    f"3GPP Records         : "
    f"{gpp_count:,}"
)

print(
    f"3GPP Skipped         : "
    f"{gpp_skipped:,}"
)

print(
    f"Total Records        : "
    f"{total_count:,}"
)

print(
    f"TCC Processing       : "
    f"{tcc_elapsed:.2f}s"
)

print(
    f"3GPP Processing      : "
    f"{gpp_elapsed:.2f}s"
)

print(
    f"Total Processing     : "
    f"{total_elapsed:.2f}s"
)


print(
    "\nRecords by Source:"
)

display(
    source_summary
)


print("=" * 80)

print(
    "✓ Source corpora normalized "
    "and merged successfully."
)

print(
    "✓ Normalized schema and "
    "record counts validated."
)


conn.close()

BUILDING NORMALIZED TELECOM CORPUS

[1/2] Normalizing full TCC corpus...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     ✓ 1,778,423 TCC records loaded | 366.05s

[2/2] Normalizing GSMA/3GPP specifications...
     Loaded 2,000 / 15,052
     Loaded 4,000 / 15,052
     Loaded 6,000 / 15,052
     Loaded 8,000 / 15,052
     Loaded 10,000 / 15,052
     Loaded 12,000 / 15,052


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     Loaded 14,000 / 15,052


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


NORMALIZED CORPUS SUMMARY
TCC Records          : 1,778,423
3GPP Records         : 15,052
3GPP Skipped         : 0
Total Records        : 1,793,475
TCC Processing       : 366.05s
3GPP Processing      : 658.72s
Total Processing     : 1051.79s

Records by Source:


,source_family,document_type,records
0,3GPP,3gpp_specification,15052
1,TCC,3gpp_tsg_document,574785
2,TCC,tcc_document,1203638


## **Cell 8 — Deduplicate + Validate**

In [ ]:
# ============================================================
# CELL 8 — DEDUPLICATE + VALIDATE NORMALIZED CORPUS
# ============================================================


# ============================================================
# DEDUPLICATION CONFIGURATION
# ============================================================

# Conservative settings retained for the targeted
# memory-efficient deduplication stage.
DEDUP_THREADS = 2
DEDUP_MEMORY_LIMIT = "20GB"


# ============================================================
# CONNECT
# ============================================================

conn = duckdb.connect(
    str(DB_PATH)
)

conn.execute(
    f"SET threads = {DEDUP_THREADS}"
)

conn.execute(
    f"SET memory_limit = "
    f"'{DEDUP_MEMORY_LIMIT}'"
)

conn.execute(
    f"SET temp_directory = "
    f"'{str(TEMP_DIR)}'"
)

conn.execute(
    "SET preserve_insertion_order = false"
)


# ============================================================
# PRE-DEDUPLICATION ANALYSIS
# ============================================================

print("=" * 80)
print("NORMALIZED CORPUS — DEDUPLICATION")
print("=" * 80)


before_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM telecom_corpus
    """
).fetchone()[0]


print(
    f"Records Before : "
    f"{before_count:,}"
)


start = (
    time.perf_counter()
)


# ============================================================
# 1. IDENTIFY EXACT-TEXT DUPLICATE HASHES
# ============================================================

conn.execute("""
    DROP TABLE IF EXISTS duplicate_hashes
""")


conn.execute("""
    CREATE TABLE duplicate_hashes AS

    SELECT
        text_hash,
        COUNT(*) AS duplicate_count

    FROM telecom_corpus

    GROUP BY text_hash

    HAVING COUNT(*) > 1
""")


duplicate_groups = conn.execute(
    """
    SELECT COUNT(*)
    FROM duplicate_hashes
    """
).fetchone()[0]


duplicate_records = conn.execute(
    """
    SELECT
        COALESCE(
            SUM(
                duplicate_count - 1
            ),
            0
        )

    FROM duplicate_hashes
    """
).fetchone()[0]


# ============================================================
# CROSS-SOURCE DUPLICATE CHECK
# ============================================================

cross_source_duplicates = conn.execute(
    """
    SELECT COUNT(*)

    FROM (

        SELECT
            c.text_hash

        FROM telecom_corpus c

        INNER JOIN duplicate_hashes d
            ON c.text_hash = d.text_hash

        GROUP BY
            c.text_hash

        HAVING COUNT(
            DISTINCT c.source_family
        ) > 1
    )
    """
).fetchone()[0]


print(
    f"Duplicate Groups          : "
    f"{duplicate_groups:,}"
)

print(
    f"Duplicate Records         : "
    f"{duplicate_records:,}"
)

print(
    f"Cross-Source Hashes       : "
    f"{cross_source_duplicates:,}"
)


duplicate_pct = (
    duplicate_records
    / before_count
    * 100

    if before_count
    else 0
)


print(
    f"Potential Reduction       : "
    f"{duplicate_pct:.2f}%"
)


# ============================================================
# DUPLICATE DISTRIBUTION BY SOURCE
# ============================================================

duplicate_source_summary = conn.execute(
    """
    SELECT
        c.source_family,
        COUNT(*) AS records_in_duplicate_groups

    FROM telecom_corpus c

    INNER JOIN duplicate_hashes d
        ON c.text_hash = d.text_hash

    GROUP BY
        c.source_family

    ORDER BY
        records_in_duplicate_groups DESC
    """
).df()


print(
    "\nDuplicate Records by Source:"
)

display(
    duplicate_source_summary
)


# ============================================================
# 2. COPY NON-DUPLICATE RECORDS
# ============================================================

conn.execute("""
    DROP TABLE IF EXISTS telecom_corpus_dedup
""")


conn.execute("""
    CREATE TABLE telecom_corpus_dedup AS

    SELECT
        c.*

    FROM telecom_corpus c

    LEFT JOIN duplicate_hashes d
        ON c.text_hash = d.text_hash

    WHERE d.text_hash IS NULL
""")


unique_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM telecom_corpus_dedup
    """
).fetchone()[0]


print(
    f"\nUnique Records Copied     : "
    f"{unique_count:,}"
)


# ============================================================
# 3. SELECT ONE COMPLETE RECORD PER DUPLICATE HASH
# ============================================================

conn.execute("""
    DROP TABLE IF EXISTS duplicate_records_keep
""")


conn.execute("""
    CREATE TABLE duplicate_records_keep AS

    SELECT
        c.*

    FROM telecom_corpus c

    INNER JOIN duplicate_hashes d
        ON c.text_hash = d.text_hash

    QUALIFY
        ROW_NUMBER() OVER (

            PARTITION BY
                c.text_hash

            ORDER BY
                c.doc_id

        ) = 1
""")


kept_duplicate_groups = conn.execute(
    """
    SELECT COUNT(*)
    FROM duplicate_records_keep
    """
).fetchone()[0]


if (
    kept_duplicate_groups
    != duplicate_groups
):

    conn.close()

    raise RuntimeError(
        "Duplicate representative count "
        "does not match duplicate groups."
    )


print(
    f"Duplicate Representatives : "
    f"{kept_duplicate_groups:,}"
)


# ============================================================
# 4. ADD DUPLICATE REPRESENTATIVES
# ============================================================

conn.execute("""
    INSERT INTO telecom_corpus_dedup

    SELECT *
    FROM duplicate_records_keep
""")


# ============================================================
# 5. VALIDATE DEDUPLICATED CORPUS
# ============================================================

after_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM telecom_corpus_dedup
    """
).fetchone()[0]


removed_count = (
    before_count
    - after_count
)


remaining_duplicates = conn.execute(
    """
    SELECT COUNT(*)

    FROM (

        SELECT
            text_hash

        FROM telecom_corpus_dedup

        GROUP BY
            text_hash

        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


invalid_text = conn.execute(
    """
    SELECT COUNT(*)

    FROM telecom_corpus_dedup

    WHERE text IS NULL

       OR length(
            trim(text)
       ) < ?
    """,
    [MIN_TEXT_LENGTH]
).fetchone()[0]


missing_hash = conn.execute(
    """
    SELECT COUNT(*)

    FROM telecom_corpus_dedup

    WHERE text_hash IS NULL
    """
).fetchone()[0]


# ============================================================
# SCHEMA VALIDATION
# ============================================================

dedup_schema = [
    row[0]

    for row in conn.execute(
        """
        DESCRIBE telecom_corpus_dedup
        """
    ).fetchall()
]


schema_valid = (
    dedup_schema
    == NORMALIZED_SCHEMA
)


# ============================================================
# COUNT VALIDATION
# ============================================================

expected_after_count = (
    before_count
    - duplicate_records
)


validation_passed = (

    remaining_duplicates == 0

    and invalid_text == 0

    and missing_hash == 0

    and schema_valid

    and removed_count
        == duplicate_records

    and after_count
        == expected_after_count
)


# ============================================================
# 6. PROMOTE ONLY AFTER SUCCESSFUL VALIDATION
# ============================================================

if not validation_passed:

    conn.close()

    raise RuntimeError(
        "Deduplication validation failed. "
        "Original telecom_corpus retained."
    )


conn.execute(
    "BEGIN TRANSACTION"
)


try:

    conn.execute("""
        DROP TABLE telecom_corpus
    """)

    conn.execute("""
        ALTER TABLE telecom_corpus_dedup
        RENAME TO telecom_corpus
    """)

    conn.execute("""
        DROP TABLE duplicate_records_keep
    """)

    conn.execute("""
        DROP TABLE duplicate_hashes
    """)

    conn.execute(
        "COMMIT"
    )


except Exception:

    conn.execute(
        "ROLLBACK"
    )

    conn.close()

    raise


conn.execute(
    "CHECKPOINT"
)


elapsed = (
    time.perf_counter()
    - start
)


# ============================================================
# FINAL CORPUS VALIDATION
# ============================================================

final_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM telecom_corpus
    """
).fetchone()[0]


final_duplicate_hashes = conn.execute(
    """
    SELECT COUNT(*)

    FROM (

        SELECT
            text_hash

        FROM telecom_corpus

        GROUP BY
            text_hash

        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


conn.close()


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("DEDUPLICATION SUMMARY")
print("=" * 80)

print(
    f"Records Before          : "
    f"{before_count:,}"
)

print(
    f"Records After           : "
    f"{final_count:,}"
)

print(
    f"Records Removed         : "
    f"{removed_count:,}"
)

print(
    f"Reduction               : "
    f"{duplicate_pct:.2f}%"
)

print(
    f"Cross-Source Hashes     : "
    f"{cross_source_duplicates:,}"
)


print("\nValidation")

print(
    f"Duplicate Hashes Left   : "
    f"{final_duplicate_hashes:,}"
)

print(
    f"Invalid Text            : "
    f"{invalid_text:,}"
)

print(
    f"Missing Text Hash       : "
    f"{missing_hash:,}"
)

print(
    f"Schema Valid            : "
    f"{schema_valid}"
)

print(
    f"Count Validation        : "
    f"{after_count == expected_after_count}"
)

print(
    f"\nElapsed Time            : "
    f"{elapsed:.2f} sec"
)

print("=" * 80)

print(
    "✓ Exact-text deduplication "
    "completed successfully."
)

print(
    "✓ Deduplicated corpus validated "
    "and promoted to telecom_corpus."
)

NORMALIZED DATABASE — PRE-DEDUPLICATION
Database Size     : 42.95 GB
Total Records              : 1,793,475
Duplicate Text Groups      : 9,647
Duplicate Records Removable: 12,537
Cross-Source Duplicate Hashes: 0
Potential Reduction        : 0.70%

Duplicate Records by Source:


,source_family,records_in_duplicate_groups
0,TCC,22184


## **Cell 9 — Telecom Tokenization Validation**


In [ ]:
# ============================================================
# CELL 9 — TELECOM TOKENIZATION VALIDATION
# ============================================================


# ============================================================
# FROZEN FTS TOKENIZATION CONFIGURATION
# ============================================================

FTS_CONFIG = {
    "stemmer": "none",
    "stopwords": "none",
    "ignore": r"[^a-z0-9.-]+",
    "lower": True,
    "strip_accents": True
}


# ============================================================
# TELECOM TOKENIZATION TEST CORPUS
# ============================================================

TOKEN_TEST_DOCS = pd.DataFrame({
    "doc_id": [
        "doc_1",
        "doc_2",
        "doc_3",
        "doc_4",
        "doc_5",
        "doc_6"
    ],

    "text": [
        "5G registration uses the AMF over the N2 interface.",

        "3GPP TS 23.501 defines the 5G system architecture.",

        "S-NSSAI and 5QI are important QoS and slicing identifiers.",

        "PFCP is used between the SMF and UPF over the N4 interface.",

        "NG-RAN supports interfaces including N2, N3, Xn and F1.",

        "GTP-U carries user-plane traffic across the N3 interface."
    ]
})


# ============================================================
# TERMS THAT MUST REMAIN SEARCHABLE
# ============================================================

TEST_TERMS = [
    "5G",
    "3GPP",
    "N2",
    "N4",
    "5QI",
    "S-NSSAI",
    "NG-RAN",
    "23.501",
    "PFCP",
    "GTP-U",
    "Xn",
    "F1"
]


# ============================================================
# CREATE TEMPORARY VALIDATION DATABASE
# ============================================================

test_conn = duckdb.connect()


test_conn.register(
    "token_test_df",
    TOKEN_TEST_DOCS
)


test_conn.execute("""
    CREATE TABLE token_test AS

    SELECT *
    FROM token_test_df
""")


# ============================================================
# LOAD DUCKDB FTS
# ============================================================

test_conn.execute(
    "INSTALL fts"
)

test_conn.execute(
    "LOAD fts"
)


# ============================================================
# BUILD FTS INDEX USING FROZEN CONFIGURATION
# ============================================================

test_conn.execute(
    f"""
    PRAGMA create_fts_index(
        'token_test',
        'doc_id',
        'text',

        stemmer = '{FTS_CONFIG["stemmer"]}',
        stopwords = '{FTS_CONFIG["stopwords"]}',
        ignore = '{FTS_CONFIG["ignore"]}',

        lower = {str(FTS_CONFIG["lower"]).lower()},
        strip_accents = {str(FTS_CONFIG["strip_accents"]).lower()}
    )
    """
)


# ============================================================
# VALIDATE TELECOM TERMS
# ============================================================

results = []


for term in TEST_TERMS:

    matches = test_conn.execute(
        """
        SELECT
            doc_id,
            score

        FROM (

            SELECT
                doc_id,

                fts_main_token_test.match_bm25(
                    doc_id,
                    ?
                ) AS score

            FROM token_test
        )

        WHERE score IS NOT NULL

        ORDER BY score DESC
        """,
        [term]
    ).df()


    results.append({
        "term":
            term,

        "matches":
            len(matches),

        "matched_docs":
            ", ".join(
                matches["doc_id"].tolist()
            )
            if not matches.empty
            else "-"
    })


# ============================================================
# RESULTS
# ============================================================

tokenization_results = (
    pd.DataFrame(
        results
    )
)


display(
    tokenization_results
)


# ============================================================
# VALIDATION
# ============================================================

failed_terms = (
    tokenization_results.loc[
        tokenization_results[
            "matches"
        ] == 0,
        "term"
    ].tolist()
)


if failed_terms:

    test_conn.close()

    raise RuntimeError(
        "FTS tokenization validation failed "
        "for telecom terms: "
        f"{failed_terms}"
    )


# ============================================================
# CLEANUP
# ============================================================

test_conn.close()


# ============================================================
# SUMMARY
# ============================================================

print("=" * 80)
print("TELECOM FTS TOKENIZATION VALIDATION")
print("=" * 80)

print(
    f"Stemmer        : "
    f"{FTS_CONFIG['stemmer']}"
)

print(
    f"Stopwords      : "
    f"{FTS_CONFIG['stopwords']}"
)

print(
    f"Ignore Pattern : "
    f"{FTS_CONFIG['ignore']}"
)

print(
    f"Lowercase      : "
    f"{FTS_CONFIG['lower']}"
)

print(
    f"Strip Accents  : "
    f"{FTS_CONFIG['strip_accents']}"
)

print(
    f"Terms Tested   : "
    f"{len(TEST_TERMS)}"
)

print(
    f"Terms Matched  : "
    f"{len(TEST_TERMS) - len(failed_terms)}"
)

print("=" * 80)

print(
    "✓ Telecom-aware FTS tokenization "
    "configuration validated."
)

TELECOM TOKENIZATION / FTS TEST


,term,matches,matched_docs
0,5G,2,"doc_2, doc_1"
1,3GPP,1,doc_2
2,N2,3,"doc_5, doc_1, doc_4"
3,N4,3,"doc_5, doc_1, doc_4"
4,5QI,1,doc_3
5,S-NSSAI,1,doc_3
6,NG-RAN,1,doc_5
7,23.501,0,-
8,PFCP,1,doc_4
9,GTP-U,0,-


## **Cell 10 — Build Persistent BM25 / FTS Shards**

In [ ]:
# ============================================================
# CELL 10 — BUILD PERSISTENT BM25 / FTS SHARDS
# ============================================================


# ============================================================
# FROZEN SHARD PLAN
# ============================================================

SHARD_PLAN = {

    # Dedicated normative 3GPP specifications
    ("3GPP", "3GPP-Specifications"): 2,

    # Telco Common Corpus collections
    ("TCC", "3GPP-TSG"): 4,
    ("TCC", "USPTO"): 4,
    ("TCC", "IEEE-Access"): 2,
    ("TCC", "IETF-Mail-Daily"): 2,
    ("TCC", "EPO"): 1,
    ("TCC", "OpenAlex"): 1,
    ("TCC", "IETF-Drafts"): 1,
    ("TCC", "IETF-RFCs"): 1,
    ("TCC", "Wikidata-Telecom"): 1,
    ("TCC", "Wikipedia-Telecom"): 1,
    ("TCC", "IETF-Proceedings"): 1
}


# ============================================================
# BUILD CONFIGURATION
# ============================================================

SHARD_DIR = (
    WORK_DIR
    / "bm25_shards"
)

SHARD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


SHARD_MANIFEST_PATH = (
    SHARD_DIR
    / "bm25_shard_manifest.csv"
)


SHARD_THREADS = 4
SHARD_MEMORY_LIMIT = "20GB"

MIN_FREE_DISK_GB = 20
MIN_COMPACTION_FREE_GB = 40

FTS_TABLE = "telecom_shard"
FTS_SCHEMA = "fts_main_telecom_shard"

COMPACT_DB_PATH = (
    DB_PATH.with_name(
        "telecom_corpus_compact.duckdb"
    )
)

COMPACTION_MARKER = (
    WORK_DIR
    / ".source_compaction_complete"
)


# ============================================================
# HELPERS
# ============================================================

def get_free_disk_gb() -> float:
    """Return available disk space in GiB."""

    _, _, free = shutil.disk_usage(
        WORK_DIR
    )

    return (
        free
        / (1024 ** 3)
    )


def get_file_size_gb(path: Path) -> float:
    """Return file size in GiB."""

    if not path.exists():
        return 0.0

    return (
        path.stat().st_size
        / (1024 ** 3)
    )


def load_fts_extension(conn):
    """Load DuckDB FTS, installing it if required."""

    try:

        conn.execute(
            "LOAD fts"
        )

    except Exception:

        conn.execute(
            "INSTALL fts"
        )

        conn.execute(
            "LOAD fts"
        )


def shard_is_complete(
    shard_path: Path,
    expected_records: int
) -> bool:
    """
    Validate an existing shard before deciding
    whether it can safely be reused.
    """

    if not shard_path.exists():
        return False


    try:

        conn = duckdb.connect(
            str(shard_path),
            read_only=True
        )


        table_exists = conn.execute(
            """
            SELECT COUNT(*)

            FROM information_schema.tables

            WHERE table_name = ?
            """,
            [FTS_TABLE]
        ).fetchone()[0] > 0


        fts_exists = conn.execute(
            """
            SELECT COUNT(*)

            FROM information_schema.schemata

            WHERE schema_name = ?
            """,
            [FTS_SCHEMA]
        ).fetchone()[0] > 0


        record_count = 0


        if table_exists:

            record_count = conn.execute(
                f"""
                SELECT COUNT(*)
                FROM {FTS_TABLE}
                """
            ).fetchone()[0]


        conn.close()


        return (
            table_exists
            and fts_exists
            and record_count == expected_records
        )


    except Exception:

        return False


def remove_partial_shard(
    shard_path: Path
):
    """Remove incomplete shard files."""

    candidates = [
        shard_path,
        Path(
            str(shard_path)
            + ".wal"
        )
    ]


    for path in candidates:

        if path.exists():
            path.unlink()


# ============================================================
# VALIDATE DEDUPLICATED SOURCE DATABASE
# ============================================================

if not DB_PATH.exists():

    raise FileNotFoundError(
        f"Source corpus database not found: "
        f"{DB_PATH}"
    )


source_conn = duckdb.connect(
    str(DB_PATH),
    read_only=True
)


SOURCE_RECORD_COUNT = source_conn.execute(
    """
    SELECT COUNT(*)
    FROM telecom_corpus
    """
).fetchone()[0]


source_duplicate_hashes = source_conn.execute(
    """
    SELECT COUNT(*)

    FROM (

        SELECT
            text_hash

        FROM telecom_corpus

        GROUP BY
            text_hash

        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


source_pairs = {
    (
        row[0],
        row[1]
    )

    for row in source_conn.execute(
        """
        SELECT DISTINCT
            source_family,
            collection

        FROM telecom_corpus
        """
    ).fetchall()
}


source_conn.close()


if source_duplicate_hashes != 0:

    raise RuntimeError(
        "Source corpus still contains "
        "duplicate text hashes."
    )


planned_pairs = set(
    SHARD_PLAN.keys()
)


missing_from_plan = (
    source_pairs
    - planned_pairs
)

unused_plan_entries = (
    planned_pairs
    - source_pairs
)


if missing_from_plan:

    raise RuntimeError(
        "Shard plan does not cover source collections: "
        f"{sorted(missing_from_plan)}"
    )


if unused_plan_entries:

    raise RuntimeError(
        "Shard plan contains collections "
        "not present in the corpus: "
        f"{sorted(unused_plan_entries)}"
    )


print("=" * 90)
print("BM25 SHARD BUILD — SOURCE VALIDATION")
print("=" * 90)

print(
    f"Source Records       : "
    f"{SOURCE_RECORD_COUNT:,}"
)

print(
    f"Duplicate Hashes     : "
    f"{source_duplicate_hashes:,}"
)

print(
    f"Collections Covered  : "
    f"{len(source_pairs)}"
)

print(
    f"Planned Shards       : "
    f"{sum(SHARD_PLAN.values())}"
)

print("=" * 90)


# ============================================================
# COMPACT SOURCE DATABASE
# ============================================================

# Deduplication creates and drops large intermediate tables.
# A fresh DuckDB copy releases those internally reusable blocks
# before the persistent shard build begins.

if not COMPACTION_MARKER.exists():

    source_size_before_gb = (
        get_file_size_gb(
            DB_PATH
        )
    )


    if (
        get_free_disk_gb()
        < MIN_COMPACTION_FREE_GB
    ):

        raise RuntimeError(
            "Insufficient free disk for source "
            "database compaction."
        )


    print(
        "\n" + "=" * 90
    )

    print(
        "COMPACTING SOURCE DATABASE"
    )

    print("=" * 90)

    print(
        f"Source Size Before   : "
        f"{source_size_before_gb:.2f} GB"
    )

    print(
        f"Free Disk Before     : "
        f"{get_free_disk_gb():.2f} GB"
    )


    if COMPACT_DB_PATH.exists():
        COMPACT_DB_PATH.unlink()


    compact_start = (
        time.perf_counter()
    )


    compact_conn = (
        duckdb.connect()
    )

    compact_conn.execute(
        "SET threads = 4"
    )

    compact_conn.execute(
        "SET memory_limit = '20GB'"
    )

    compact_conn.execute(
        "SET preserve_insertion_order = false"
    )


    source_sql = (
        str(DB_PATH)
        .replace(
            "'",
            "''"
        )
    )

    target_sql = (
        str(COMPACT_DB_PATH)
        .replace(
            "'",
            "''"
        )
    )


    compact_conn.execute(
        f"""
        ATTACH
            '{source_sql}'
        AS src
        (READ_ONLY)
        """
    )

    compact_conn.execute(
        f"""
        ATTACH
            '{target_sql}'
        AS dst
        """
    )

    compact_conn.execute(
        """
        COPY FROM DATABASE
            src
        TO
            dst
        """
    )

    compact_conn.execute(
        "DETACH dst"
    )

    compact_conn.execute(
        "DETACH src"
    )

    compact_conn.close()


    compact_elapsed = (
        time.perf_counter()
        - compact_start
    )


    # --------------------------------------------------------
    # VALIDATE COMPACT COPY
    # --------------------------------------------------------

    check_conn = duckdb.connect(
        str(COMPACT_DB_PATH),
        read_only=True
    )


    compact_count = check_conn.execute(
        """
        SELECT COUNT(*)
        FROM telecom_corpus
        """
    ).fetchone()[0]


    compact_duplicates = check_conn.execute(
        """
        SELECT COUNT(*)

        FROM (

            SELECT
                text_hash

            FROM telecom_corpus

            GROUP BY
                text_hash

            HAVING COUNT(*) > 1
        )
        """
    ).fetchone()[0]


    compact_schema = [
        row[0]

        for row in check_conn.execute(
            """
            DESCRIBE telecom_corpus
            """
        ).fetchall()
    ]


    check_conn.close()


    compact_valid = (
        compact_count
        == SOURCE_RECORD_COUNT

        and compact_duplicates == 0

        and compact_schema
        == NORMALIZED_SCHEMA
    )


    if not compact_valid:

        if COMPACT_DB_PATH.exists():
            COMPACT_DB_PATH.unlink()

        raise RuntimeError(
            "Compact source database "
            "failed validation."
        )


    # --------------------------------------------------------
    # PROMOTE COMPACT COPY
    # --------------------------------------------------------

    backup_path = (
        DB_PATH.with_name(
            "telecom_corpus_precompact.duckdb"
        )
    )


    if backup_path.exists():
        backup_path.unlink()


    DB_PATH.rename(
        backup_path
    )

    COMPACT_DB_PATH.rename(
        DB_PATH
    )

    backup_path.unlink()


    COMPACTION_MARKER.write_text(
        "validated",
        encoding="utf-8"
    )


    source_size_after_gb = (
        get_file_size_gb(
            DB_PATH
        )
    )


    print(
        f"Source Size After    : "
        f"{source_size_after_gb:.2f} GB"
    )

    print(
        f"Space Reclaimed      : "
        f"{source_size_before_gb - source_size_after_gb:.2f} GB"
    )

    print(
        f"Compaction Time      : "
        f"{compact_elapsed / 60:.2f} min"
    )

    print(
        f"Free Disk After      : "
        f"{get_free_disk_gb():.2f} GB"
    )

    print(
        "✓ Compact source database "
        "validated and promoted."
    )


else:

    print(
        "\n✓ Source compaction already "
        "completed in this build session."
    )


# ============================================================
# BUILD DETERMINISTIC SHARD MANIFEST
# ============================================================

source_conn = duckdb.connect(
    str(DB_PATH),
    read_only=True
)


manifest_rows = []


for (
    source_family,
    collection
), shard_count in SHARD_PLAN.items():


    shard_stats = source_conn.execute(
        """
        SELECT
            hash(doc_id) % ? AS shard_id,

            COUNT(*) AS records,

            COALESCE(
                SUM(
                    length(text)
                ),
                0
            ) AS text_chars

        FROM telecom_corpus

        WHERE source_family = ?
          AND collection = ?

        GROUP BY
            shard_id

        ORDER BY
            shard_id
        """,
        [
            shard_count,
            source_family,
            collection
        ]
    ).fetchall()


    stat_lookup = {
        int(row[0]): (
            int(row[1]),
            int(row[2])
        )

        for row in shard_stats
    }


    for shard_id in range(
        shard_count
    ):

        records, text_chars = (
            stat_lookup.get(
                shard_id,
                (0, 0)
            )
        )


        shard_name = (
            source_family.lower()
            + "_"
            + collection.lower()
                .replace(
                    "-",
                    "_"
                )
            + "_"
            + str(
                shard_id
            ).zfill(2)
        )


        manifest_rows.append({

            "shard_name":
                shard_name,

            "source_family":
                source_family,

            "collection":
                collection,

            "shard_id":
                shard_id,

            "shards_in_collection":
                shard_count,

            "records":
                records,

            "text_gb":
                text_chars
                / (1024 ** 3)
        })


source_conn.close()


SHARD_MANIFEST = pd.DataFrame(
    manifest_rows
)


manifest_record_count = int(
    SHARD_MANIFEST[
        "records"
    ].sum()
)


if (
    manifest_record_count
    != SOURCE_RECORD_COUNT
):

    raise RuntimeError(
        "Shard manifest does not cover "
        "the complete source corpus."
    )


if (
    len(SHARD_MANIFEST)
    != 21
):

    raise RuntimeError(
        "Frozen Version B architecture "
        "must contain 21 shards."
    )


print(
    "\n" + "=" * 90
)

print(
    "DETERMINISTIC BM25 SHARD PLAN"
)

print("=" * 90)


display(
    SHARD_MANIFEST[
        [
            "shard_name",
            "source_family",
            "collection",
            "records",
            "text_gb"
        ]
    ].round({
        "text_gb": 2
    })
)


print(
    f"\nTotal Shards          : "
    f"{len(SHARD_MANIFEST)}"
)

print(
    f"Indexed Records       : "
    f"{manifest_record_count:,}"
)

print(
    f"Approx. Indexed Text  : "
    f"{SHARD_MANIFEST['text_gb'].sum():.2f} GB"
)

print(
    f"Largest Shard         : "
    f"{SHARD_MANIFEST['text_gb'].max():.2f} GB"
)

print("=" * 90)


# ============================================================
# BUILD + VALIDATE BM25 SHARDS
# ============================================================

build_results = []

total_shards = (
    len(SHARD_MANIFEST)
)


print(
    "\n" + "=" * 90
)

print(
    "PERSISTENT BM25 SHARD BUILD"
)

print("=" * 90)

print(
    f"Source Database      : "
    f"{DB_PATH}"
)

print(
    f"Shard Directory      : "
    f"{SHARD_DIR}"
)

print(
    f"Total Shards         : "
    f"{total_shards}"
)

print(
    f"Threads / Shard      : "
    f"{SHARD_THREADS}"
)

print(
    f"Memory Limit         : "
    f"{SHARD_MEMORY_LIMIT}"
)

print(
    f"Free Disk            : "
    f"{get_free_disk_gb():.2f} GB"
)

print("=" * 90)


# ============================================================
# FROZEN FTS SETTINGS FROM CELL 9
# ============================================================

fts_stemmer = (
    FTS_CONFIG[
        "stemmer"
    ]
)

fts_stopwords = (
    FTS_CONFIG[
        "stopwords"
    ]
)

fts_ignore = (
    FTS_CONFIG[
        "ignore"
    ].replace(
        "'",
        "''"
    )
)

fts_lower = str(
    FTS_CONFIG[
        "lower"
    ]
).lower()

fts_strip_accents = str(
    FTS_CONFIG[
        "strip_accents"
    ]
).lower()


# ============================================================
# BUILD SHARDS SEQUENTIALLY
# ============================================================

for shard_number, row in (
    SHARD_MANIFEST.iterrows()
):

    shard_name = (
        row["shard_name"]
    )

    source_family = (
        row["source_family"]
    )

    collection = (
        row["collection"]
    )

    shard_id = int(
        row["shard_id"]
    )

    shard_count = int(
        row[
            "shards_in_collection"
        ]
    )

    expected_records = int(
        row["records"]
    )

    expected_text_gb = float(
        row["text_gb"]
    )


    shard_path = (
        SHARD_DIR
        / f"{shard_name}.duckdb"
    )


    print(
        "\n" + "=" * 90
    )

    print(
        f"SHARD "
        f"{shard_number + 1}/"
        f"{total_shards}"
        f" — "
        f"{shard_name}"
    )

    print("=" * 90)

    print(
        f"Collection          : "
        f"{collection}"
    )

    print(
        f"Expected Records    : "
        f"{expected_records:,}"
    )

    print(
        f"Expected Text       : "
        f"{expected_text_gb:.2f} GB"
    )

    print(
        f"Free Disk           : "
        f"{get_free_disk_gb():.2f} GB"
    )


    # --------------------------------------------------------
    # REUSE VALIDATED EXISTING SHARD
    # --------------------------------------------------------

    if shard_is_complete(
        shard_path,
        expected_records
    ):

        print(
            "✓ Existing validated shard "
            "found — reusing."
        )


        build_results.append({

            "shard_name":
                shard_name,

            "source_family":
                source_family,

            "collection":
                collection,

            "shard_id":
                shard_id,

            "shards_in_collection":
                shard_count,

            "records":
                expected_records,

            "text_gb":
                expected_text_gb,

            "db_size_gb":
                get_file_size_gb(
                    shard_path
                ),

            "table_build_sec":
                0.0,

            "fts_build_sec":
                0.0,

            "total_build_sec":
                0.0,

            "status":
                "EXISTING",

            "filename":
                shard_path.name
        })


        pd.DataFrame(
            build_results
        ).to_csv(
            SHARD_MANIFEST_PATH,
            index=False
        )


        continue


    # --------------------------------------------------------
    # DISK SAFETY
    # --------------------------------------------------------

    if (
        get_free_disk_gb()
        < MIN_FREE_DISK_GB
    ):

        raise RuntimeError(
            "Shard build stopped because "
            f"free disk dropped below "
            f"{MIN_FREE_DISK_GB} GB."
        )


    remove_partial_shard(
        shard_path
    )


    # --------------------------------------------------------
    # SHARD-SPECIFIC TEMP SPACE
    # --------------------------------------------------------

    shard_temp_dir = (
        TEMP_DIR
        / shard_name
    )


    if shard_temp_dir.exists():

        shutil.rmtree(
            shard_temp_dir
        )


    shard_temp_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    shard_total_start = (
        time.perf_counter()
    )


    shard_conn = None


    try:

        # ----------------------------------------------------
        # CREATE SHARD DATABASE
        # ----------------------------------------------------

        shard_conn = (
            duckdb.connect(
                str(shard_path)
            )
        )


        shard_conn.execute(
            f"SET threads = "
            f"{SHARD_THREADS}"
        )

        shard_conn.execute(
            f"SET memory_limit = "
            f"'{SHARD_MEMORY_LIMIT}'"
        )

        shard_conn.execute(
            "SET preserve_insertion_order = false"
        )


        temp_sql = (
            str(shard_temp_dir)
            .replace(
                "'",
                "''"
            )
        )


        shard_conn.execute(
            f"SET temp_directory = "
            f"'{temp_sql}'"
        )


        # ----------------------------------------------------
        # ATTACH SOURCE DATABASE
        # ----------------------------------------------------

        source_sql = (
            str(DB_PATH)
            .replace(
                "'",
                "''"
            )
        )


        shard_conn.execute(
            f"""
            ATTACH
                '{source_sql}'
            AS source_db
            (READ_ONLY)
            """
        )


        # ----------------------------------------------------
        # MATERIALIZE SHARD
        # ----------------------------------------------------

        print(
            "\n[1/3] "
            "Creating shard table..."
        )


        table_start = (
            time.perf_counter()
        )


        shard_conn.execute(
            """
            CREATE TABLE telecom_shard AS

            SELECT *

            FROM source_db.telecom_corpus

            WHERE source_family = ?

              AND collection = ?

              AND (
                    hash(doc_id)
                    % ?
                  ) = ?
            """,
            [
                source_family,
                collection,
                shard_count,
                shard_id
            ]
        )


        table_build_sec = (
            time.perf_counter()
            - table_start
        )


        actual_records = (
            shard_conn.execute(
                """
                SELECT COUNT(*)
                FROM telecom_shard
                """
            ).fetchone()[0]
        )


        if (
            actual_records
            != expected_records
        ):

            raise RuntimeError(
                f"{shard_name}: expected "
                f"{expected_records:,} records "
                f"but built "
                f"{actual_records:,}."
            )


        print(
            f"     ✓ "
            f"{actual_records:,} records"
            f" | "
            f"{table_build_sec:.2f}s"
        )


        shard_conn.execute(
            "DETACH source_db"
        )


        # ----------------------------------------------------
        # BUILD TELECOM-AWARE BM25 INDEX
        # ----------------------------------------------------

        load_fts_extension(
            shard_conn
        )


        print(
            "[2/3] "
            "Building BM25 / FTS index..."
        )


        fts_start = (
            time.perf_counter()
        )


        shard_conn.execute(
            f"""
            PRAGMA create_fts_index(
                'telecom_shard',
                'doc_id',

                'identifier',
                'title',
                'text',

                stemmer =
                    '{fts_stemmer}',

                stopwords =
                    '{fts_stopwords}',

                ignore =
                    '{fts_ignore}',

                strip_accents =
                    {fts_strip_accents},

                lower =
                    {fts_lower}
            )
            """
        )


        fts_build_sec = (
            time.perf_counter()
            - fts_start
        )


        # ----------------------------------------------------
        # VALIDATE FTS
        # ----------------------------------------------------

        fts_ready = (
            shard_conn.execute(
                """
                SELECT COUNT(*)

                FROM information_schema.schemata

                WHERE schema_name = ?
                """,
                [FTS_SCHEMA]
            ).fetchone()[0]
            > 0
        )


        if not fts_ready:

            raise RuntimeError(
                f"{shard_name}: "
                "FTS validation failed."
            )


        # ----------------------------------------------------
        # CHECKPOINT
        # ----------------------------------------------------

        print(
            "[3/3] "
            "Checkpointing shard..."
        )


        shard_conn.execute(
            "CHECKPOINT"
        )


        shard_conn.close()
        shard_conn = None


    except Exception:

        if shard_conn is not None:

            try:
                shard_conn.close()

            except Exception:
                pass


        remove_partial_shard(
            shard_path
        )

        raise


    finally:

        if shard_temp_dir.exists():

            shutil.rmtree(
                shard_temp_dir,
                ignore_errors=True
            )


    # --------------------------------------------------------
    # RECORD COMPLETED SHARD
    # --------------------------------------------------------

    shard_total_sec = (
        time.perf_counter()
        - shard_total_start
    )


    shard_size_gb = (
        get_file_size_gb(
            shard_path
        )
    )


    print(
        f"\n✓ {shard_name} COMPLETE"
    )

    print(
        f"  Records       : "
        f"{actual_records:,}"
    )

    print(
        f"  Table Build   : "
        f"{table_build_sec:.2f}s"
    )

    print(
        f"  FTS Build     : "
        f"{fts_build_sec:.2f}s"
    )

    print(
        f"  Total         : "
        f"{shard_total_sec:.2f}s"
    )

    print(
        f"  Database Size : "
        f"{shard_size_gb:.2f} GB"
    )

    print(
        f"  Free Disk     : "
        f"{get_free_disk_gb():.2f} GB"
    )


    build_results.append({

        "shard_name":
            shard_name,

        "source_family":
            source_family,

        "collection":
            collection,

        "shard_id":
            shard_id,

        "shards_in_collection":
            shard_count,

        "records":
            actual_records,

        "text_gb":
            expected_text_gb,

        "db_size_gb":
            shard_size_gb,

        "table_build_sec":
            table_build_sec,

        "fts_build_sec":
            fts_build_sec,

        "total_build_sec":
            shard_total_sec,

        "status":
            "COMPLETE",

        "filename":
            shard_path.name
    })


    # Save progress after every completed shard.
    pd.DataFrame(
        build_results
    ).to_csv(
        SHARD_MANIFEST_PATH,
        index=False
    )


# ============================================================
# FINAL BUILD VALIDATION
# ============================================================

BUILD_RESULTS = pd.DataFrame(
    build_results
)


built_records = int(
    BUILD_RESULTS[
        "records"
    ].sum()
)


all_shards_valid = all(

    shard_is_complete(
        SHARD_DIR
        / f"{row['shard_name']}.duckdb",

        int(
            row["records"]
        )
    )

    for _, row
    in BUILD_RESULTS.iterrows()
)


build_valid = (
    len(BUILD_RESULTS)
        == total_shards

    and built_records
        == SOURCE_RECORD_COUNT

    and all_shards_valid
)


if not build_valid:

    raise RuntimeError(
        "Final BM25 shard build "
        "validation failed."
    )


# Save complete final manifest.
BUILD_RESULTS.to_csv(
    SHARD_MANIFEST_PATH,
    index=False
)


# ============================================================
# BUILD SUMMARY
# ============================================================

total_shard_size_gb = (
    BUILD_RESULTS[
        "db_size_gb"
    ].sum()
)

total_fts_time = (
    BUILD_RESULTS[
        "fts_build_sec"
    ].sum()
)

total_build_time = (
    BUILD_RESULTS[
        "total_build_sec"
    ].sum()
)


print(
    "\n" + "=" * 90
)

print(
    "BM25 SHARD BUILD SUMMARY"
)

print("=" * 90)


display(
    BUILD_RESULTS[
        [
            "shard_name",
            "collection",
            "records",
            "db_size_gb",
            "status"
        ]
    ].round({
        "db_size_gb": 2
    })
)


print(
    f"\nValidated Shards      : "
    f"{len(BUILD_RESULTS)}"
    f" / "
    f"{total_shards}"
)

print(
    f"Indexed Records       : "
    f"{built_records:,}"
)

print(
    f"Total Shard Storage   : "
    f"{total_shard_size_gb:.2f} GB"
)

print(
    f"FTS Build Time        : "
    f"{total_fts_time / 60:.2f} min"
)

print(
    f"Total Build Time      : "
    f"{total_build_time / 60:.2f} min"
)

print(
    f"Free Disk Remaining   : "
    f"{get_free_disk_gb():.2f} GB"
)

print(
    f"Manifest              : "
    f"{SHARD_MANIFEST_PATH}"
)

print("=" * 90)

print(
    "✓ All persistent BM25 shards "
    "built and validated successfully."
)

PERSISTENT BM25 SHARD BUILD
Source Database      : /tmp/telecom_bm25/telecom_corpus.duckdb
Shard Directory      : /tmp/telecom_bm25/bm25_shards
Total Shards         : 21
Threads / Shard      : 4
Memory Limit         : 20GB
Free Disk            : 155.57 GB

SHARD 1/21 — tcc_3gpp_tsg_00
Collection          : 3GPP-TSG
Expected Records    : 143,679
Expected Text       : 2.60 GB
Free Disk           : 155.57 GB

[1/3] Creating shard table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     ✓ 143,679 records | 291.97s
[2/3] Building BM25 / FTS index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[3/3] Checkpointing shard...

✓ 3gpp_3gpp_specifications_01 COMPLETE
  Records           : 7,531
  Table Build       : 15.17s
  FTS Build         : 546.72s
  Total             : 562.64s
  Database Size     : 2.67 GB
  Free Disk         : 86.56 GB

BM25 SHARD BUILD SUMMARY


,shard_name,status,source_family,collection,shard_id,records,text_gb,db_size_gb,table_build_sec,fts_build_sec,total_build_sec
0,tcc_3gpp_tsg_00,COMPLETE,TCC,3GPP-TSG,0,143679,2.604165,4.978283,291.970525,657.030029,950.568171
1,tcc_3gpp_tsg_01,COMPLETE,TCC,3GPP-TSG,1,142683,2.615533,4.998791,248.344972,620.415443,870.332542
2,tcc_3gpp_tsg_02,COMPLETE,TCC,3GPP-TSG,2,143196,2.582649,4.932873,223.358490,665.198632,890.328504
3,tcc_3gpp_tsg_03,COMPLETE,TCC,3GPP-TSG,3,143114,2.586078,4.945080,226.683533,664.398513,892.707512
4,tcc_uspto_00,COMPLETE,TCC,USPTO,0,23986,2.387394,4.180431,258.990572,684.704070,945.156011
5,tcc_uspto_01,COMPLETE,TCC,USPTO,1,23888,2.379336,4.171642,209.419490,769.284474,980.217573
6,tcc_uspto_02,COMPLETE,TCC,USPTO,2,23714,2.380670,4.224133,209.680504,693.691121,905.315167
7,tcc_uspto_03,COMPLETE,TCC,USPTO,3,23716,2.365075,4.140881,205.159984,680.908810,887.684195
8,tcc_ieee_access_00,COMPLETE,TCC,IEEE-Access,0,32109,2.051079,4.100842,307.281205,581.088480,893.484488
9,tcc_ieee_access_01,COMPLETE,TCC,IEEE-Access,1,31932,2.036881,4.072277,208.146232,613.927294,824.390810



Completed / Existing Shards : 21 / 21
Total Shard Storage         : 69.00 GB
Total FTS Build Time        : 170.74 min
Total Build Time            : 242.47 min
Free Disk Remaining         : 86.56 GB
Manifest                    : /tmp/telecom_bm25/bm25_shard_manifest.csv


**Observation — BM25 Shard Build**

* The monolithic BM25 build exceeded available temporary disk capacity and failed.
* The full corpus was therefore partitioned into **21 deterministic BM25 shards**.
* All **21/21 shards completed successfully**, covering **1,780,938 records**.
* Total persistent BM25 storage was **69.00 GB**, with a total build time of **242.47 minutes**.
* Temporary build storage was released after each shard, preventing cumulative disk exhaustion.

***Key Decision:*** Freeze the **21-shard BM25 architecture** for Version B and retain the failed monolithic build as a documented architectural learning.


## **Cell 11 — Persist BM25 Artifact to Kaggle**

In [ ]:
# ============================================================
# CELL 11 — PERSIST BM25 ARTIFACT TO KAGGLE
# ============================================================

import json
import subprocess


# ============================================================
# PERSISTENCE CONFIGURATION
# ============================================================

# False:
#   Validate the locally built artifact but skip Kaggle upload.
#
# True:
#   Authenticate using KAGGLE_API_TOKEN from Colab Secrets and
#   create/update the persisted Kaggle dataset.

UPLOAD_TO_KAGGLE = False


KAGGLE_DATASET = (
    "cliffordimaguezegie/"
    "telecom-bm25-indexed-knowledge-base"
)

KAGGLE_DATASET_TITLE = (
    "Telecom BM25 Indexed Knowledge Base"
)

KAGGLE_VERSION_MESSAGE = (
    "Rebuilt Version B telecom BM25 knowledge base"
)


# ============================================================
# CELL 11A — PREPARE + VALIDATE BM25 ARTIFACT
# ============================================================

KAGGLE_CHECKPOINT_DIR = (
    SHARD_DIR
)

checkpoint_manifest = (
    SHARD_MANIFEST_PATH
)


shard_files = sorted(
    KAGGLE_CHECKPOINT_DIR.glob(
        "*.duckdb"
    )
)


total_size_gb = sum(
    path.stat().st_size
    for path in shard_files
) / (1024 ** 3)


expected_shards = len(
    SHARD_MANIFEST
)


print("=" * 80)
print("KAGGLE BM25 ARTIFACT PREPARATION")
print("=" * 80)

print(
    f"Artifact Directory : "
    f"{KAGGLE_CHECKPOINT_DIR}"
)

print(
    f"Expected Shards    : "
    f"{expected_shards}"
)

print(
    f"DuckDB Shards      : "
    f"{len(shard_files)}"
)

print(
    f"Shard Storage      : "
    f"{total_size_gb:.2f} GB"
)

print(
    f"Manifest Present   : "
    f"{checkpoint_manifest.exists()}"
)

print("=" * 80)


artifact_ready = (

    len(shard_files)
    == expected_shards

    and checkpoint_manifest.exists()
)


if not artifact_ready:

    raise RuntimeError(
        "BM25 artifact is incomplete. "
        "Persistence validation failed."
    )


print(
    "✓ BM25 artifact ready "
    "for persistence."
)


# ============================================================
# CELL 11B — WRITE KAGGLE DATASET METADATA
# ============================================================

metadata_path = (
    KAGGLE_CHECKPOINT_DIR
    / "dataset-metadata.json"
)


metadata = {

    "title":
        KAGGLE_DATASET_TITLE,

    "id":
        KAGGLE_DATASET,

    "licenses": [
        {
            "name":
                "other"
        }
    ]
}


with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=2
    )


if not metadata_path.exists():

    raise RuntimeError(
        "Kaggle dataset metadata "
        "could not be created."
    )


print(
    f"✓ Kaggle metadata ready: "
    f"{metadata_path}"
)


# ============================================================
# CELL 11C — UPLOAD CONTROL
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "KAGGLE PERSISTENCE CONFIGURATION"
)

print("=" * 80)

print(
    f"Upload Enabled : "
    f"{UPLOAD_TO_KAGGLE}"
)

print(
    f"Dataset        : "
    f"{KAGGLE_DATASET}"
)

print("=" * 80)


if not UPLOAD_TO_KAGGLE:

    print(
        "✓ Kaggle upload skipped for "
        "this notebook run."
    )

    print(
        "✓ Local BM25 artifact remains "
        "available for downstream validation."
    )


else:

    # ========================================================
    # CELL 11D — KAGGLE AUTHENTICATION FROM COLAB SECRET
    # ========================================================

    try:

        kaggle_api_token = (
            userdata.get(
                "KAGGLE_API_TOKEN"
            )
        )

    except Exception as exc:

        raise RuntimeError(
            "Unable to read KAGGLE_API_TOKEN "
            "from Google Colab Secrets."
        ) from exc


    if not kaggle_api_token:

        raise RuntimeError(
            "UPLOAD_TO_KAGGLE=True but "
            "KAGGLE_API_TOKEN was not found "
            "in Google Colab Secrets."
        )


    os.environ[
        "KAGGLE_API_TOKEN"
    ] = kaggle_api_token


    print(
        "✓ KAGGLE_API_TOKEN loaded "
        "from Colab Secrets."
    )


    # ========================================================
    # CELL 11E — VERIFY KAGGLE CLI
    # ========================================================

    version_check = subprocess.run(
        [
            "kaggle",
            "--version"
        ],
        capture_output=True,
        text=True
    )


    if version_check.returncode != 0:

        raise RuntimeError(
            "Kaggle CLI is unavailable. "
            "Check the Cell 0 installation."
        )


    print(
        f"✓ {version_check.stdout.strip()}"
    )


    # ========================================================
    # CELL 11F — CHECK WHETHER DATASET ALREADY EXISTS
    # ========================================================

    dataset_check = subprocess.run(
        [
            "kaggle",
            "datasets",
            "files",
            KAGGLE_DATASET,
            "--page-size",
            "50"
        ],
        capture_output=True,
        text=True
    )


    dataset_exists = (
        dataset_check.returncode
        == 0
    )


    print(
        f"Dataset Exists : "
        f"{dataset_exists}"
    )


    # ========================================================
    # CELL 11G — CREATE OR VERSION DATASET
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "PERSISTING BM25 ARTIFACT TO KAGGLE"
    )

    print("=" * 80)

    print(
        f"Source Directory : "
        f"{KAGGLE_CHECKPOINT_DIR}"
    )

    print(
        f"Dataset          : "
        f"{KAGGLE_DATASET}"
    )

    print(
        f"DuckDB Shards    : "
        f"{len(shard_files)}"
    )

    print(
        f"Artifact Size    : "
        f"{total_size_gb:.2f} GB"
    )

    print(
        "Visibility       : PRIVATE"
    )

    print("=" * 80)


    if dataset_exists:

        print(
            "Existing Kaggle dataset detected."
        )

        print(
            "Creating a new dataset version..."
        )


        upload_result = subprocess.run(
            [
                "kaggle",
                "datasets",
                "version",

                "-p",
                str(
                    KAGGLE_CHECKPOINT_DIR
                ),

                "-m",
                KAGGLE_VERSION_MESSAGE,

                "-r",
                "skip"
            ],
            text=True
        )


    else:

        print(
            "Dataset does not yet exist."
        )

        print(
            "Creating Kaggle dataset..."
        )


        upload_result = subprocess.run(
            [
                "kaggle",
                "datasets",
                "create",

                "-p",
                str(
                    KAGGLE_CHECKPOINT_DIR
                ),

                "-r",
                "skip"
            ],
            text=True
        )


    if upload_result.returncode != 0:

        raise RuntimeError(
            "Kaggle dataset persistence failed."
        )


    # ========================================================
    # CELL 11H — VERIFY DATASET STATUS
    # ========================================================

    status_result = subprocess.run(
        [
            "kaggle",
            "datasets",
            "status",
            KAGGLE_DATASET
        ],
        capture_output=True,
        text=True
    )


    if status_result.returncode != 0:

        raise RuntimeError(
            "Unable to verify Kaggle "
            "dataset status.\n\n"
            f"{status_result.stderr}"
        )


    print(
        "\n" + "=" * 80
    )

    print(
        "KAGGLE DATASET STATUS"
    )

    print("=" * 80)

    print(
        status_result.stdout.strip()
    )


    # ========================================================
    # CELL 11I — VERIFY DATASET FILES
    # ========================================================

    files_result = subprocess.run(
        [
            "kaggle",
            "datasets",
            "files",
            KAGGLE_DATASET,
            "--page-size",
            "50"
        ],
        capture_output=True,
        text=True
    )


    if files_result.returncode != 0:

        raise RuntimeError(
            "Unable to verify persisted "
            "Kaggle dataset files.\n\n"
            f"{files_result.stderr}"
        )


    print(
        "\n" + "=" * 80
    )

    print(
        "KAGGLE DATASET FILES"
    )

    print("=" * 80)

    print(
        files_result.stdout
    )


    print(
        "✓ BM25 artifact successfully "
        "persisted to Kaggle."
    )

KAGGLE BM25 CHECKPOINT PREPARATION
Checkpoint Directory : /tmp/telecom_bm25/bm25_shards
DuckDB Shards        : 21
Shard Storage        : 69.00 GB
Manifest Present     : True
✓ Checkpoint directory ready for Kaggle upload.


## **Cell 12 - Database Health + Size + Build Report**

In [ ]:
# ============================================================
# CELL 12 — DATABASE HEALTH + SIZE + BUILD REPORT
# ============================================================


# ============================================================
# EXPECTED BUILD STATE
# ============================================================

EXPECTED_SHARDS = (
    len(SHARD_MANIFEST)
)

EXPECTED_RECORDS = (
    SOURCE_RECORD_COUNT
)


# ============================================================
# VALIDATE SOURCE DATABASE
# ============================================================

if not DB_PATH.exists():

    raise FileNotFoundError(
        f"Source database not found: "
        f"{DB_PATH}"
    )


source_conn = duckdb.connect(
    str(DB_PATH),
    read_only=True
)


source_records = source_conn.execute(
    """
    SELECT COUNT(*)
    FROM telecom_corpus
    """
).fetchone()[0]


source_duplicates = source_conn.execute(
    """
    SELECT COUNT(*)

    FROM (

        SELECT
            text_hash

        FROM telecom_corpus

        GROUP BY
            text_hash

        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


source_schema = [
    row[0]

    for row in source_conn.execute(
        """
        DESCRIBE telecom_corpus
        """
    ).fetchall()
]


source_storage = source_conn.execute(
    """
    PRAGMA database_size
    """
).df()


source_conn.close()


source_schema_valid = (
    source_schema
    == NORMALIZED_SCHEMA
)


source_db_size_gb = (
    DB_PATH.stat().st_size
    / (1024 ** 3)
)


# ============================================================
# VALIDATE PERSISTED BUILD MANIFEST
# ============================================================

if not SHARD_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "BM25 shard manifest was not found: "
        f"{SHARD_MANIFEST_PATH}"
    )


persisted_manifest = pd.read_csv(
    SHARD_MANIFEST_PATH
)


required_manifest_columns = {
    "shard_name",
    "source_family",
    "collection",
    "shard_id",
    "shards_in_collection",
    "records",
    "db_size_gb",
    "status"
}


missing_manifest_columns = (
    required_manifest_columns
    - set(
        persisted_manifest.columns
    )
)


if missing_manifest_columns:

    raise RuntimeError(
        "Shard manifest is missing required "
        "columns: "
        f"{sorted(missing_manifest_columns)}"
    )


manifest_records = int(
    persisted_manifest[
        "records"
    ].sum()
)


manifest_shards = len(
    persisted_manifest
)


# ============================================================
# VALIDATE EVERY BM25 SHARD
# ============================================================

health_results = []


print("=" * 90)
print("BM25 SHARD HEALTH VALIDATION")
print("=" * 90)


for _, row in SHARD_MANIFEST.iterrows():

    shard_name = (
        row["shard_name"]
    )

    expected_records = int(
        row["records"]
    )


    shard_path = (
        SHARD_DIR
        / f"{shard_name}.duckdb"
    )


    result = {

        "shard_name":
            shard_name,

        "file_exists":
            shard_path.exists(),

        "expected_records":
            expected_records,

        "actual_records":
            0,

        "table_ready":
            False,

        "fts_ready":
            False,

        "size_gb":
            0.0,

        "status":
            "FAILED"
    }


    # --------------------------------------------------------
    # FILE CHECK
    # --------------------------------------------------------

    if not shard_path.exists():

        health_results.append(
            result
        )

        continue


    result["size_gb"] = (
        shard_path.stat().st_size
        / (1024 ** 3)
    )


    # --------------------------------------------------------
    # DATABASE + FTS CHECK
    # --------------------------------------------------------

    try:

        conn = duckdb.connect(
            str(shard_path),
            read_only=True
        )


        result["table_ready"] = (
            conn.execute(
                """
                SELECT COUNT(*)

                FROM information_schema.tables

                WHERE table_name = ?
                """,
                [FTS_TABLE]
            ).fetchone()[0]
            > 0
        )


        result["fts_ready"] = (
            conn.execute(
                """
                SELECT COUNT(*)

                FROM information_schema.schemata

                WHERE schema_name = ?
                """,
                [FTS_SCHEMA]
            ).fetchone()[0]
            > 0
        )


        if result["table_ready"]:

            result["actual_records"] = (
                conn.execute(
                    f"""
                    SELECT COUNT(*)
                    FROM {FTS_TABLE}
                    """
                ).fetchone()[0]
            )


        conn.close()


        if (
            result["table_ready"]
            and result["fts_ready"]
            and result[
                "actual_records"
            ] == expected_records
        ):

            result["status"] = "PASS"


    except Exception as exc:

        result["status"] = (
            f"ERROR: {exc}"
        )


    health_results.append(
        result
    )


# ============================================================
# BUILD HEALTH DATAFRAME
# ============================================================

SHARD_HEALTH = pd.DataFrame(
    health_results
)


passed_shards = int(
    SHARD_HEALTH[
        "status"
    ].eq(
        "PASS"
    ).sum()
)


actual_total_records = int(
    SHARD_HEALTH[
        "actual_records"
    ].sum()
)


total_shard_size_gb = (
    SHARD_HEALTH[
        "size_gb"
    ].sum()
)


# ============================================================
# FILESET VALIDATION
# ============================================================

actual_shard_files = sorted(
    SHARD_DIR.glob(
        "*.duckdb"
    )
)


actual_shard_file_count = (
    len(actual_shard_files)
)


# ============================================================
# DISK STATE
# ============================================================

_, used_disk, free_disk = (
    shutil.disk_usage(
        WORK_DIR
    )
)


used_disk_gb = (
    used_disk
    / (1024 ** 3)
)


free_disk_gb = (
    free_disk
    / (1024 ** 3)
)


# ============================================================
# OVERALL VALIDATION
# ============================================================

source_valid = (

    source_records
    == EXPECTED_RECORDS

    and source_duplicates == 0

    and source_schema_valid
)


manifest_valid = (

    manifest_shards
    == EXPECTED_SHARDS

    and manifest_records
    == EXPECTED_RECORDS
)


shards_valid = (

    passed_shards
    == EXPECTED_SHARDS

    and actual_total_records
    == EXPECTED_RECORDS

    and actual_shard_file_count
    == EXPECTED_SHARDS
)


BUILD_HEALTHY = (

    source_valid

    and manifest_valid

    and shards_valid
)


# ============================================================
# SHARD REPORT
# ============================================================

display(
    SHARD_HEALTH[
        [
            "shard_name",
            "expected_records",
            "actual_records",
            "fts_ready",
            "size_gb",
            "status"
        ]
    ].round({
        "size_gb": 2
    })
)


# ============================================================
# FINAL BUILD REPORT
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "VERSION B — FINAL BM25 BUILD HEALTH REPORT"
)

print("=" * 90)


print("\nSOURCE DATABASE")

print(
    f"Records               : "
    f"{source_records:,}"
)

print(
    f"Duplicate Hashes      : "
    f"{source_duplicates:,}"
)

print(
    f"Schema Valid          : "
    f"{source_schema_valid}"
)

print(
    f"Database Size         : "
    f"{source_db_size_gb:.2f} GB"
)


print("\nBM25 SHARDS")

print(
    f"Validated Shards      : "
    f"{passed_shards}"
    f" / "
    f"{EXPECTED_SHARDS}"
)

print(
    f"Indexed Records       : "
    f"{actual_total_records:,}"
)

print(
    f"Total Indexed Storage : "
    f"{total_shard_size_gb:.2f} GB"
)


print("\nMANIFEST")

print(
    f"Manifest Shards       : "
    f"{manifest_shards}"
)

print(
    f"Manifest Records      : "
    f"{manifest_records:,}"
)

print(
    f"Manifest Path         : "
    f"{SHARD_MANIFEST_PATH}"
)


print("\nDISK")

print(
    f"Disk Used             : "
    f"{used_disk_gb:.2f} GB"
)

print(
    f"Disk Free             : "
    f"{free_disk_gb:.2f} GB"
)


print("=" * 90)


if not BUILD_HEALTHY:

    raise RuntimeError(
    "Final BM25 build health validation failed. "
    "Investigate the build before using the artifact."
)


print(
    "✓ Source database passed integrity validation."
)

print(
    "✓ All BM25 shards passed integrity validation."
)

print(
    "✓ Build manifest matches the indexed corpus."
)

print(
    "✓ Version B BM25 knowledge base passed "
    "final build health validation."
)

# **SECTION 3 — Retrieval Validation + Optimization**

## **Cell 13 — Baseline Global BM25 Retrieval**

In [ ]:
# ============================================================
# CELL 13 — BASELINE GLOBAL BM25 RETRIEVAL
# ============================================================

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)


# ============================================================
# BASELINE RETRIEVAL CONFIGURATION
# ============================================================

BASELINE_TOP_K = 5
BASELINE_TOP_K_PER_SHARD = 5

BASELINE_SEARCH_WORKERS = 2

BM25_K = 1.2
BM25_B = 0.75

FTS_SCHEMA = (
    "fts_main_telecom_shard"
)


# ============================================================
# DISCOVER PERSISTED BM25 SHARDS
# ============================================================

SHARD_FILES = sorted(
    SHARD_DIR.glob(
        "*.duckdb"
    )
)


if (
    len(SHARD_FILES)
    != len(SHARD_MANIFEST)
):

    raise RuntimeError(
        "BM25 shard discovery does not match "
        "the build manifest."
    )


# ============================================================
# QUERY PROCESSING
# ============================================================

QUERY_STOPWORDS = {
    "a", "an", "the", "and", "or",
    "of", "to", "for", "in", "on",
    "with", "by", "from", "as", "at",
    "what", "which", "how", "why",
    "when", "where", "is", "are",
    "was", "were", "be", "been",
    "do", "does", "did",
    "describe", "explain"
}


def telecom_tokenize(text: str) -> list[str]:
    """
    Preserve telecom identifiers such as:
    5G, N2, S-NSSAI, NG-RAN, 23.501 and GTP-U.
    """

    return re.findall(
        r"[A-Za-z0-9]+"
        r"(?:[.-][A-Za-z0-9]+)*",
        (text or "").lower()
    )


def process_query(
    query: str
) -> list[str]:
    """
    Convert a natural-language query into
    searchable telecom terms.
    """

    return [
        term

        for term
        in telecom_tokenize(
            query
        )

        if term
        not in QUERY_STOPWORDS
    ]


# ============================================================
# GLOBAL CORPUS STATISTICS
# ============================================================

def get_global_corpus_stats(
    shard_files
) -> tuple:
    """
    Calculate global document count and weighted
    average document length across disjoint shards.
    """

    stats_rows = []


    for shard_path in shard_files:

        conn = duckdb.connect(
            str(shard_path),
            read_only=True
        )

        conn.execute(
            "SET threads = 1"
        )

        conn.execute(
            "LOAD fts"
        )


        num_docs, avgdl = (
            conn.execute(
                f"""
                SELECT
                    num_docs,
                    avgdl

                FROM {FTS_SCHEMA}.stats
                """
            ).fetchone()
        )


        conn.close()


        stats_rows.append({
            "num_docs":
                int(num_docs),

            "avgdl":
                float(avgdl)
        })


    stats_df = pd.DataFrame(
        stats_rows
    )


    global_num_docs = int(
        stats_df[
            "num_docs"
        ].sum()
    )


    global_avgdl = (
        (
            stats_df[
                "num_docs"
            ]
            *
            stats_df[
                "avgdl"
            ]
        ).sum()
        / global_num_docs
    )


    return (
        global_num_docs,
        global_avgdl
    )


# ============================================================
# GLOBAL QUERY-TERM DOCUMENT FREQUENCIES
# ============================================================

def get_global_term_df(
    shard_files,
    query_terms
) -> dict:
    """
    Sum document frequencies for query terms
    across the disjoint BM25 shards.
    """

    df_map = {
        term: 0
        for term in query_terms
    }


    if not query_terms:
        return df_map


    placeholders = ",".join(
        ["?"] * len(
            query_terms
        )
    )


    for shard_path in shard_files:

        conn = duckdb.connect(
            str(shard_path),
            read_only=True
        )

        conn.execute(
            "SET threads = 1"
        )

        conn.execute(
            "LOAD fts"
        )


        rows = conn.execute(
            f"""
            SELECT
                term,
                df

            FROM {FTS_SCHEMA}.dict

            WHERE term IN (
                {placeholders}
            )
            """,
            query_terms
        ).fetchall()


        conn.close()


        for term, df in rows:

            df_map[
                term
            ] += int(df)


    return df_map


# ============================================================
# CACHE GLOBAL DOCUMENT STATISTICS
# ============================================================

stats_start = (
    time.perf_counter()
)


(
    GLOBAL_NUM_DOCS,
    GLOBAL_AVGDL
) = get_global_corpus_stats(
    SHARD_FILES
)


stats_elapsed = (
    time.perf_counter()
    - stats_start
)


if (
    GLOBAL_NUM_DOCS
    != SOURCE_RECORD_COUNT
):

    raise RuntimeError(
        "Global BM25 document count does not "
        "match the source corpus."
    )


print("=" * 90)
print("GLOBAL BM25 CORPUS STATISTICS")
print("=" * 90)

print(
    f"Shard Files      : "
    f"{len(SHARD_FILES)}"
)

print(
    f"Global Documents : "
    f"{GLOBAL_NUM_DOCS:,}"
)

print(
    f"Global AvgDL     : "
    f"{GLOBAL_AVGDL:,.2f}"
)

print(
    f"Statistics Time  : "
    f"{stats_elapsed:.3f} sec"
)

print("=" * 90)


# ============================================================
# SEARCH ONE SHARD USING DIRECT INVERTED-INDEX BM25
# ============================================================

def search_baseline_shard(
    shard_path: Path,
    query_terms: list[str],
    global_df_map: dict,
    top_k_per_shard: int
):
    """
    Search one shard using direct access to the
    DuckDB FTS inverted-index tables.
    """

    shard_start = (
        time.perf_counter()
    )


    active_terms = [
        term

        for term
        in query_terms

        if global_df_map.get(
            term,
            0
        ) > 0
    ]


    if not active_terms:

        return (
            shard_path.stem,
            pd.DataFrame(),
            0.0
        )


    conn = duckdb.connect(
        str(shard_path),
        read_only=True
    )

    conn.execute(
        "SET threads = 1"
    )

    conn.execute(
        "SET memory_limit = '4GB'"
    )

    conn.execute(
        "LOAD fts"
    )


    values_sql = ", ".join(
        ["(?, ?)"]
        * len(active_terms)
    )


    parameters = []


    for term in active_terms:

        parameters.extend([
            term,
            int(
                global_df_map[
                    term
                ]
            )
        ])


    sql = f"""
        WITH

        global_query_terms(
            term,
            global_df
        ) AS (

            VALUES
                {values_sql}
        ),

        qtermids AS (

            SELECT
                d.termid,
                d.term,
                q.global_df

            FROM {FTS_SCHEMA}.dict d

            INNER JOIN global_query_terms q
                ON d.term = q.term
        ),

        qterms AS (

            SELECT
                t.docid,
                t.termid

            FROM {FTS_SCHEMA}.terms t

            INNER JOIN qtermids q
                ON t.termid = q.termid
        ),

        term_tf AS (

            SELECT
                docid,
                termid,
                COUNT(*) AS tf

            FROM qterms

            GROUP BY
                docid,
                termid
        ),

        subscores AS (

            SELECT
                tf.docid,
                tf.termid,

                LOG(
                    (
                        (
                            (
                                {GLOBAL_NUM_DOCS}
                                - q.global_df
                                + 0.5
                            )
                            /
                            (
                                q.global_df
                                + 0.5
                            )
                        )
                        + 1
                    )
                )

                *

                (
                    tf.tf
                    * ({BM25_K} + 1)
                )

                /

                (
                    tf.tf
                    +
                    (
                        {BM25_K}
                        *
                        (
                            (1 - {BM25_B})
                            +
                            (
                                {BM25_B}
                                *
                                (
                                    d.len
                                    / {GLOBAL_AVGDL}
                                )
                            )
                        )
                    )
                )

                AS bm25_subscore

            FROM term_tf tf

            INNER JOIN {FTS_SCHEMA}.docs d
                ON tf.docid = d.docid

            INNER JOIN qtermids q
                ON tf.termid = q.termid
        ),

        scores AS (

            SELECT
                docid,

                SUM(
                    bm25_subscore
                ) AS bm25_score,

                COUNT(
                    DISTINCT termid
                ) AS matched_terms

            FROM subscores

            GROUP BY
                docid

            ORDER BY
                bm25_score DESC

            LIMIT
                {top_k_per_shard}
        )

        SELECT
            c.doc_id,
            c.source,
            c.source_family,
            c.collection,
            c.identifier,
            c.title,
            c.release,
            c.version,
            c.date,
            c.document_type,
            c.source_path,

            s.matched_terms,
            s.bm25_score

        FROM scores s

        INNER JOIN {FTS_SCHEMA}.docs d
            ON s.docid = d.docid

        INNER JOIN telecom_shard c
            ON c.doc_id = d.name

        ORDER BY
            s.bm25_score DESC
    """


    results = conn.execute(
        sql,
        parameters
    ).df()


    conn.close()


    elapsed = (
        time.perf_counter()
        - shard_start
    )


    if not results.empty:

        results[
            "shard_name"
        ] = shard_path.stem


    return (
        shard_path.stem,
        results,
        elapsed
    )


# ============================================================
# BASELINE GLOBAL BM25 SEARCH
# ============================================================

def search_baseline_bm25(
    query: str,
    top_k: int = BASELINE_TOP_K
):
    """
    Search all 21 BM25 shards as one unrestricted
    logical corpus using globally comparable BM25 scores.
    """

    start = (
        time.perf_counter()
    )


    query_terms = (
        process_query(
            query
        )
    )


    if not query_terms:

        raise ValueError(
            "Query contains no searchable terms."
        )


    global_df_map = (
        get_global_term_df(
            SHARD_FILES,
            query_terms
        )
    )


    shard_results = []
    shard_timings = []
    shard_errors = []


    print("=" * 90)
    print("BASELINE GLOBAL BM25 SEARCH")
    print("=" * 90)

    print(
        f"Original Query : "
        f"{query}"
    )

    print(
        f"Query Terms    : "
        f"{query_terms}"
    )

    print(
        f"Shards         : "
        f"{len(SHARD_FILES)}"
    )

    print(
        f"Workers        : "
        f"{BASELINE_SEARCH_WORKERS}"
    )

    print("=" * 90)


    with ThreadPoolExecutor(
        max_workers=
            BASELINE_SEARCH_WORKERS
    ) as executor:


        futures = {

            executor.submit(
                search_baseline_shard,
                shard_path,
                query_terms,
                global_df_map,
                BASELINE_TOP_K_PER_SHARD
            ):
                shard_path.stem

            for shard_path
            in SHARD_FILES
        }


        for future in as_completed(
            futures
        ):

            shard_name = (
                futures[
                    future
                ]
            )


            try:

                (
                    _,
                    results,
                    elapsed
                ) = future.result()


                shard_timings.append({
                    "shard_name":
                        shard_name,

                    "search_sec":
                        elapsed,

                    "results":
                        len(
                            results
                        )
                })


                if not results.empty:

                    shard_results.append(
                        results
                    )


            except Exception as exc:

                shard_errors.append({
                    "shard_name":
                        shard_name,

                    "error":
                        str(exc)
                })


    if shard_errors:

        raise RuntimeError(
            "Baseline BM25 search failed "
            f"on {len(shard_errors)} shard(s): "
            f"{shard_errors}"
        )


    if not shard_results:

        return (
            pd.DataFrame(),
            pd.DataFrame(),
            time.perf_counter()
            - start
        )


    candidates = pd.concat(
        shard_results,
        ignore_index=True
    )


    final_results = (

        candidates

        .sort_values(
            [
                "bm25_score",
                "matched_terms"
            ],
            ascending=[
                False,
                False
            ]
        )

        .drop_duplicates(
            subset="doc_id"
        )

        .head(
            top_k
        )

        .reset_index(
            drop=True
        )
    )


    final_results.insert(
        0,
        "rank",
        range(
            1,
            len(final_results) + 1
        )
    )


    total_elapsed = (
        time.perf_counter()
        - start
    )


    timing_df = (
        pd.DataFrame(
            shard_timings
        )
    )


    return (
        final_results,
        timing_df,
        total_elapsed
    )


# ============================================================
# BASELINE RETRIEVAL TEST
# ============================================================

TEST_QUERY = (
    "What are the primary responsibilities "
    "of the AMF in a 5G Standalone network?"
)


(
    baseline_results,
    baseline_timings,
    baseline_latency
) = search_baseline_bm25(
    TEST_QUERY,
    top_k=5
)


# ============================================================
# BASELINE REPORT
# ============================================================

print("\n" + "=" * 90)
print("BASELINE BM25 RETRIEVAL SUMMARY")
print("=" * 90)

print(
    f"Shards Searched : "
    f"{len(baseline_timings)}"
    f" / "
    f"{len(SHARD_FILES)}"
)

print(
    f"Final Results   : "
    f"{len(baseline_results)}"
)

print(
    f"Total Latency   : "
    f"{baseline_latency:.3f} sec"
)


if not baseline_timings.empty:

    print(
        f"Mean Shard Time : "
        f"{baseline_timings['search_sec'].mean():.3f} sec"
    )

    print(
        f"Slowest Shard   : "
        f"{baseline_timings['search_sec'].max():.3f} sec"
    )


print("=" * 90)


if not baseline_results.empty:

    display(
        baseline_results[
            [
                "rank",
                "source_family",
                "collection",
                "identifier",
                "title",
                "shard_name",
                "matched_terms",
                "bm25_score"
            ]
        ].round({
            "bm25_score": 4
        })
    )

PARALLEL SHARDED BM25 SEARCH
Original Query : What are the primary responsibilities of the AMF in a 5G Standalone network?
Search Query   : primary responsibilities amf 5g standalone network
Query Terms    : ['primary', 'responsibilities', 'amf', '5g', 'standalone', 'network']


NameError: name 'SHARD_MANIFEST' is not defined

**Observation — Baseline Global BM25 Retrieval**

All 21 shards were searchable as one logical corpus using global BM25 statistics, but the unrestricted baseline required approximately **129.22 seconds** for the AMF test query and did not inherently prefer the most authoritative source.

***Key Decision:*** Keep the indexes unchanged and reduce query-time work by narrowing the searched shard set. This is a build-time baseline, not the final runtime router.


## **Cell 14 — Optimized Retrieval + Source Profiles**

In [ ]:
# ============================================================
# CELL 14 — OPTIMIZED RETRIEVAL + SOURCE PROFILES
# ============================================================

import math


# ============================================================
# RETRIEVAL CONFIGURATION
# ============================================================

MAX_CANDIDATE_TERMS = 3
MAX_SEARCH_WORKERS = 6

OPTIMIZED_TOP_K = 5


# ============================================================
# SOURCE / SEARCH PROFILES
# ============================================================

SEARCH_PROFILES = {

    "all": sorted(
        SHARD_MANIFEST[
            "shard_name"
        ].tolist()
    ),

    "3gpp": sorted(
        SHARD_MANIFEST.loc[
            (
                SHARD_MANIFEST[
                    "source_family"
                ] == "3GPP"
            )
            |
            (
                SHARD_MANIFEST[
                    "collection"
                ] == "3GPP-TSG"
            ),
            "shard_name"
        ].tolist()
    ),

    "ietf": sorted(
        SHARD_MANIFEST.loc[
            SHARD_MANIFEST[
                "collection"
            ].str.startswith(
                "IETF-",
                na=False
            ),
            "shard_name"
        ].tolist()
    ),

    "research": sorted(
        SHARD_MANIFEST.loc[
            SHARD_MANIFEST[
                "collection"
            ].isin([
                "IEEE-Access",
                "OpenAlex"
            ]),
            "shard_name"
        ].tolist()
    ),

    "patents": sorted(
        SHARD_MANIFEST.loc[
            SHARD_MANIFEST[
                "collection"
            ].isin([
                "USPTO",
                "EPO"
            ]),
            "shard_name"
        ].tolist()
    )
}


# ============================================================
# PROFILE VALIDATION
# ============================================================

EXPECTED_PROFILE_COUNTS = {
    "all": 21,
    "3gpp": 6,
    "ietf": 5,
    "research": 3,
    "patents": 5
}


print("=" * 80)
print("BM25 SEARCH PROFILES")
print("=" * 80)


for profile, shard_names in SEARCH_PROFILES.items():

    expected = (
        EXPECTED_PROFILE_COUNTS[
            profile
        ]
    )

    actual = len(
        shard_names
    )

    print(
        f"{profile:<10} : "
        f"{actual} shard(s)"
    )


    if actual != expected:

        raise RuntimeError(
            f"Profile '{profile}' contains "
            f"{actual} shards; "
            f"expected {expected}."
        )


print("=" * 80)

print(
    "✓ Source profiles validated."
)


# ============================================================
# PROFILE-WIDE BM25 STATISTICS
# ============================================================

def get_profile_stats(
    shard_names,
    query_terms
):
    """
    Calculate document count, weighted average
    document length and query-term document
    frequencies across selected shards.
    """

    total_docs = 0
    weighted_dl = 0.0

    df_map = {
        term: 0
        for term in query_terms
    }


    placeholders = (
        ",".join(
            ["?"] * len(
                query_terms
            )
        )
        if query_terms
        else None
    )


    for shard_name in shard_names:

        shard_path = (
            SHARD_DIR
            / f"{shard_name}.duckdb"
        )


        conn = duckdb.connect(
            str(shard_path),
            read_only=True
        )

        conn.execute(
            "SET threads = 1"
        )

        conn.execute(
            "LOAD fts"
        )


        num_docs, avgdl = (
            conn.execute(
                f"""
                SELECT
                    num_docs,
                    avgdl

                FROM {FTS_SCHEMA}.stats
                """
            ).fetchone()
        )


        total_docs += int(
            num_docs
        )

        weighted_dl += (
            float(num_docs)
            * float(avgdl)
        )


        if query_terms:

            rows = conn.execute(
                f"""
                SELECT
                    term,
                    df

                FROM {FTS_SCHEMA}.dict

                WHERE term IN (
                    {placeholders}
                )
                """,
                query_terms
            ).fetchall()


            for term, df in rows:

                df_map[
                    term
                ] += int(df)


        conn.close()


    profile_avgdl = (
        weighted_dl
        / total_docs

        if total_docs
        else 0.0
    )


    return (
        total_docs,
        profile_avgdl,
        df_map
    )


# ============================================================
# SELECT HIGH-IDF QUERY TERMS
# ============================================================

def select_candidate_terms(
    query_terms,
    df_map,
    num_docs,
    max_terms=MAX_CANDIDATE_TERMS
):
    """
    Rank query terms by IDF and retain only
    the most discriminative terms.
    """

    scored_terms = []


    for term in query_terms:

        df = df_map.get(
            term,
            0
        )


        if df <= 0:
            continue


        idf = math.log(
            (
                (
                    num_docs
                    - df
                    + 0.5
                )
                /
                (
                    df
                    + 0.5
                )
            )
            + 1
        )


        scored_terms.append(
            (
                term,
                df,
                idf
            )
        )


    scored_terms.sort(
        key=lambda item:
            item[2],
        reverse=True
    )


    selected_terms = [
        term

        for term, _, _
        in scored_terms[
            :max_terms
        ]
    ]


    return (
        selected_terms,
        scored_terms
    )


# ============================================================
# SEARCH ONE PROFILE SHARD
# ============================================================

def search_filtered_shard(
    shard_name,
    query_terms,
    num_docs,
    avgdl,
    df_map,
    top_k=OPTIMIZED_TOP_K
):
    """
    Search one selected shard using direct BM25
    scoring over the FTS inverted index.
    """

    shard_path = (
        SHARD_DIR
        / f"{shard_name}.duckdb"
    )


    active_terms = [
        term

        for term in query_terms

        if df_map.get(
            term,
            0
        ) > 0
    ]


    if not active_terms:

        return pd.DataFrame()


    values_sql = ", ".join(
        ["(?, ?)"]
        * len(active_terms)
    )


    parameters = []


    for term in active_terms:

        parameters.extend([
            term,
            int(
                df_map[
                    term
                ]
            )
        ])


    conn = duckdb.connect(
        str(shard_path),
        read_only=True
    )

    conn.execute(
        "SET threads = 1"
    )

    conn.execute(
        "SET memory_limit = '4GB'"
    )

    conn.execute(
        "LOAD fts"
    )


    sql = f"""
        WITH

        profile_query_terms(
            term,
            global_df
        ) AS (

            VALUES
                {values_sql}
        ),

        qtermids AS (

            SELECT
                d.termid,
                d.term,
                q.global_df

            FROM {FTS_SCHEMA}.dict d

            INNER JOIN profile_query_terms q
                ON d.term = q.term
        ),

        qterms AS (

            SELECT
                t.docid,
                t.termid

            FROM {FTS_SCHEMA}.terms t

            INNER JOIN qtermids q
                ON t.termid = q.termid
        ),

        term_tf AS (

            SELECT
                docid,
                termid,
                COUNT(*) AS tf

            FROM qterms

            GROUP BY
                docid,
                termid
        ),

        subscores AS (

            SELECT
                tf.docid,
                tf.termid,

                LOG(
                    (
                        (
                            (
                                {num_docs}
                                - q.global_df
                                + 0.5
                            )
                            /
                            (
                                q.global_df
                                + 0.5
                            )
                        )
                        + 1
                    )
                )

                *

                (
                    tf.tf
                    * ({BM25_K} + 1)
                )

                /

                (
                    tf.tf
                    +
                    (
                        {BM25_K}
                        *
                        (
                            (1 - {BM25_B})
                            +
                            (
                                {BM25_B}
                                *
                                (
                                    d.len
                                    / {avgdl}
                                )
                            )
                        )
                    )
                )

                AS bm25_subscore

            FROM term_tf tf

            INNER JOIN {FTS_SCHEMA}.docs d
                ON tf.docid = d.docid

            INNER JOIN qtermids q
                ON tf.termid = q.termid
        ),

        scores AS (

            SELECT
                docid,

                SUM(
                    bm25_subscore
                ) AS bm25_score,

                COUNT(
                    DISTINCT termid
                ) AS matched_terms

            FROM subscores

            GROUP BY
                docid

            ORDER BY
                bm25_score DESC

            LIMIT
                {top_k}
        )

        SELECT
            c.doc_id,
            c.source,
            c.source_family,
            c.collection,
            c.identifier,
            c.title,
            c.release,
            c.version,
            c.date,
            c.document_type,
            c.source_path,

            s.matched_terms,
            s.bm25_score

        FROM scores s

        INNER JOIN {FTS_SCHEMA}.docs d
            ON s.docid = d.docid

        INNER JOIN telecom_shard c
            ON c.doc_id = d.name

        ORDER BY
            s.bm25_score DESC
    """


    results = conn.execute(
        sql,
        parameters
    ).df()


    conn.close()


    if not results.empty:

        results[
            "shard_name"
        ] = shard_name


    return results


# ============================================================
# FETCH TEXT ONLY AFTER FINAL TOP-K
# ============================================================

def fetch_result_text(
    shard_name,
    doc_id,
    max_chars=5000
):
    """
    Retrieve document text only after ranking is complete.
    """

    shard_path = (
        SHARD_DIR
        / f"{shard_name}.duckdb"
    )


    conn = duckdb.connect(
        str(shard_path),
        read_only=True
    )


    row = conn.execute(
        """
        SELECT
            LEFT(text, ?)

        FROM telecom_shard

        WHERE doc_id = ?
        """,
        [
            max_chars,
            doc_id
        ]
    ).fetchone()


    conn.close()


    return (
        row[0]
        if row
        else ""
    )


# ============================================================
# FINAL OPTIMIZED SEARCH
# ============================================================

def run_optimized_search(
    query,
    profile="3gpp",
    top_k=OPTIMIZED_TOP_K
):
    """
    Optimized Version B BM25 retrieval:

    1. Route to a source profile.
    2. Calculate profile-wide BM25 statistics.
    3. Select the top-3 high-IDF terms.
    4. Search selected shards in parallel.
    5. Merge globally comparable BM25 results.
    6. Fetch text only for the final Top-K.
    """

    if profile not in SEARCH_PROFILES:

        raise ValueError(
            f"Unknown search profile: "
            f"{profile}"
        )


    start = (
        time.perf_counter()
    )


    query_terms = (
        process_query(
            query
        )
    )


    if not query_terms:

        raise ValueError(
            "Query contains no searchable terms."
        )


    shard_names = (
        SEARCH_PROFILES[
            profile
        ]
    )


    # --------------------------------------------------------
    # PROFILE STATISTICS
    # --------------------------------------------------------

    (
        profile_docs,
        profile_avgdl,
        profile_df_map
    ) = get_profile_stats(
        shard_names,
        query_terms
    )


    # --------------------------------------------------------
    # TOP-3 HIGH-IDF TERMS
    # --------------------------------------------------------

    (
        candidate_terms,
        _
    ) = select_candidate_terms(
        query_terms,
        profile_df_map,
        profile_docs,
        max_terms=
            MAX_CANDIDATE_TERMS
    )


    if not candidate_terms:

        return (
            pd.DataFrame(),
            [],
            time.perf_counter()
            - start
        )


    # --------------------------------------------------------
    # ONE WORKER PER SHARD, CAPPED AT SIX
    # --------------------------------------------------------

    workers = min(
        MAX_SEARCH_WORKERS,
        len(shard_names)
    )


    results = []
    search_errors = []


    with ThreadPoolExecutor(
        max_workers=workers
    ) as executor:


        futures = {

            executor.submit(
                search_filtered_shard,
                shard_name,
                candidate_terms,
                profile_docs,
                profile_avgdl,
                profile_df_map,
                top_k
            ):
                shard_name

            for shard_name
            in shard_names
        }


        for future in as_completed(
            futures
        ):

            shard_name = (
                futures[
                    future
                ]
            )


            try:

                result = (
                    future.result()
                )


                if not result.empty:

                    results.append(
                        result
                    )


            except Exception as exc:

                search_errors.append({
                    "shard_name":
                        shard_name,

                    "error":
                        str(exc)
                })


    if search_errors:

        raise RuntimeError(
            "Optimized retrieval failed "
            f"on {len(search_errors)} shard(s): "
            f"{search_errors}"
        )


    if not results:

        return (
            pd.DataFrame(),
            candidate_terms,
            time.perf_counter()
            - start
        )


    # --------------------------------------------------------
    # GLOBAL MERGE
    # --------------------------------------------------------

    candidates = pd.concat(
        results,
        ignore_index=True
    )


    final_results = (

        candidates

        .sort_values(
            [
                "bm25_score",
                "matched_terms"
            ],
            ascending=[
                False,
                False
            ]
        )

        .drop_duplicates(
            subset="doc_id"
        )

        .head(
            top_k
        )

        .reset_index(
            drop=True
        )
    )


    final_results.insert(
        0,
        "rank",
        range(
            1,
            len(final_results) + 1
        )
    )


    # --------------------------------------------------------
    # FETCH TEXT ONLY FOR FINAL RESULTS
    # --------------------------------------------------------

    final_results[
        "text_preview"
    ] = final_results.apply(
        lambda row:
            fetch_result_text(
                row[
                    "shard_name"
                ],
                row[
                    "doc_id"
                ]
            ),
        axis=1
    )


    elapsed = (
        time.perf_counter()
        - start
    )


    return (
        final_results,
        candidate_terms,
        elapsed
    )


# ============================================================
# OPTIMIZED RETRIEVAL TEST
# ============================================================

TEST_QUERY = (
    "What are the primary responsibilities "
    "of the AMF in a 5G Standalone network?"
)


(
    optimized_results,
    candidate_terms,
    optimized_latency
) = run_optimized_search(
    query=TEST_QUERY,
    profile="3gpp",
    top_k=5
)


# ============================================================
# REPORT
# ============================================================

print("\n" + "=" * 90)
print("OPTIMIZED BM25 RETRIEVAL")
print("=" * 90)

print(
    f"Profile         : 3gpp"
)

print(
    f"Selected Shards : "
    f"{len(SEARCH_PROFILES['3gpp'])}"
)

print(
    f"Candidate Terms : "
    f"{candidate_terms}"
)

print(
    f"Workers         : "
    f"{min(MAX_SEARCH_WORKERS, len(SEARCH_PROFILES['3gpp']))}"
)

print(
    f"Latency         : "
    f"{optimized_latency:.3f} sec"
)


if (
    "baseline_latency"
    in globals()
    and optimized_latency > 0
):

    print(
        f"Baseline        : "
        f"{baseline_latency:.3f} sec"
    )

    print(
        f"Improvement     : "
        f"{baseline_latency / optimized_latency:.2f}x"
    )


print("=" * 90)


if not optimized_results.empty:

    display(
        optimized_results[
            [
                "rank",
                "source_family",
                "collection",
                "identifier",
                "title",
                "shard_name",
                "matched_terms",
                "bm25_score"
            ]
        ].round({
            "bm25_score": 4
        })
    )

FILTERED BM25 SEARCH
Query          : What are the primary responsibilities of the AMF in a 5G Standalone network?
Profile        : 3gpp
Query Terms    : ['primary', 'responsibilities', 'amf', '5g', 'standalone', 'network']
Selected Shards: 6
Workers        : 2
Profile Docs   : 587,724
Profile AvgDL  : 3,753.50


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


FILTERED SEARCH SUMMARY
Shards Searched : 6
Candidates      : 30
Statistics Time : 1.109 sec
Total Latency   : 17.881 sec


,rank,source_family,collection,identifier,title,shard_name,matched_terms,bm25_score
0,1,TCC,3GPP-TSG,R2-2601868,R2-2601868,tcc_3gpp_tsg_02,6,10.3574
1,2,TCC,3GPP-TSG,R3-261190,R3-261190,tcc_3gpp_tsg_02,6,10.2767
2,3,TCC,3GPP-TSG,C1-213101,C1-213101,tcc_3gpp_tsg_03,5,9.5282
3,4,TCC,3GPP-TSG,S3-231748,S3-231748,tcc_3gpp_tsg_03,5,9.0859
4,5,TCC,3GPP-TSG,S4-200894,S4-200894,tcc_3gpp_tsg_01,5,8.9933


**Observation — Cell 14: Build-Time Retrieval Optimization**

Relevant-shard restriction, controlled parallelism and discriminative high-IDF terms materially reduced the baseline retrieval cost and demonstrated that shard selection is the dominant optimization lever.

***Key Decision:*** Preserve this as architectural evidence, but do **not** describe the historical `source/profile + top-3 terms` implementation as the final Benchmark v2 runtime. The frozen runtime later adopted the Version A-aligned deterministic **3GPP / TCC / Hybrid router**, richer candidate/evidence focus logic and no LLM-facing profile control.


## **Cell 15 — Retrieval Quality + Latency Evaluation**

In [ ]:
# ============================================================
# CELL 15 — RETRIEVAL QUALITY + LATENCY EVALUATION
# ============================================================


# ============================================================
# RETRIEVAL TEST SET
# ============================================================

RETRIEVAL_TESTS = [

    {
        "id": "Q1",

        "query": (
            "What are the primary responsibilities "
            "of the AMF in a 5G Standalone network?"
        ),

        "profile": "3gpp",

        "expected_concepts": [
            "amf",
            "registration",
            "mobility",
            "authentication"
        ]
    },

    {
        "id": "Q2",

        "query": (
            "How does PFCP operate over the N4 "
            "interface between the SMF and UPF?"
        ),

        "profile": "3gpp",

        "expected_concepts": [
            "pfcp",
            "n4",
            "smf",
            "upf"
        ]
    },

    {
        "id": "Q3",

        "query": (
            "What is the purpose of the Xn "
            "interface in NG-RAN?"
        ),

        "profile": "3gpp",

        "expected_concepts": [
            "xn",
            "ng-ran",
            "handover"
        ]
    },

    {
        "id": "Q4",

        "query": (
            "What is the QUIC transport protocol "
            "and how is it specified?"
        ),

        "profile": "ietf",

        "expected_concepts": [
            "quic",
            "transport",
            "connection"
        ]
    }
]


# ============================================================
# QUALITY EVALUATION
# ============================================================

benchmark_rows = []

retrieval_outputs = {}


print("=" * 90)
print("RETRIEVAL QUALITY + LATENCY EVALUATION")
print("=" * 90)


for test in RETRIEVAL_TESTS:

    print(
        f"\nRunning "
        f"{test['id']} "
        f"({test['profile']})..."
    )


    (
        results,
        candidate_terms,
        latency
    ) = run_optimized_search(

        query=test["query"],

        profile=test["profile"],

        top_k=OPTIMIZED_TOP_K
    )


    retrieval_outputs[
        test["id"]
    ] = results


    # ========================================================
    # CONCEPT COVERAGE
    # ========================================================

    if not results.empty:

        combined_text = " ".join(

            (
                results[
                    "title"
                ].fillna("")

                + " "

                + results[
                    "text_preview"
                ].fillna("")
            ).tolist()

        ).lower()

    else:

        combined_text = ""


    concept_hits = [

        concept

        for concept
        in test[
            "expected_concepts"
        ]

        if concept.lower()
        in combined_text
    ]


    expected_concept_count = len(
        test[
            "expected_concepts"
        ]
    )


    concept_coverage = (

        len(
            concept_hits
        )
        / expected_concept_count

        if expected_concept_count
        else 0.0
    )


    # ========================================================
    # TOP-K DIVERSITY / DUPLICATION CHECK
    # ========================================================

    unique_docs = (

        results[
            "doc_id"
        ].nunique()

        if not results.empty
        else 0
    )


    # ========================================================
    # RECORD RESULT
    # ========================================================

    benchmark_rows.append({

        "test_id":
            test["id"],

        "profile":
            test["profile"],

        "candidate_terms":
            ", ".join(
                candidate_terms
            ),

        "latency_sec":
            latency,

        "results":
            len(results),

        "concept_hits":
            ", ".join(
                concept_hits
            ),

        "concept_coverage_pct":
            concept_coverage * 100,

        "unique_top5":
            unique_docs
    })


# ============================================================
# BENCHMARK DATAFRAME
# ============================================================

RETRIEVAL_BENCHMARK = (
    pd.DataFrame(
        benchmark_rows
    )
)


# ============================================================
# RESULTS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "RETRIEVAL BENCHMARK RESULTS"
)

print("=" * 90)


display(
    RETRIEVAL_BENCHMARK.round({
        "latency_sec": 3,
        "concept_coverage_pct": 1
    })
)


# ============================================================
# AGGREGATE SUMMARY
# ============================================================

mean_latency = (
    RETRIEVAL_BENCHMARK[
        "latency_sec"
    ].mean()
)


median_latency = (
    RETRIEVAL_BENCHMARK[
        "latency_sec"
    ].median()
)


mean_concept_coverage = (
    RETRIEVAL_BENCHMARK[
        "concept_coverage_pct"
    ].mean()
)


mean_unique_top5 = (
    RETRIEVAL_BENCHMARK[
        "unique_top5"
    ].mean()
)


print(
    "\n" + "=" * 90
)

print(
    "RETRIEVAL QUALITY SUMMARY"
)

print("=" * 90)


print(
    f"Queries Tested        : "
    f"{len(RETRIEVAL_BENCHMARK)}"
)

print(
    f"Mean Latency          : "
    f"{mean_latency:.3f} sec"
)

print(
    f"Median Latency        : "
    f"{median_latency:.3f} sec"
)

print(
    f"Mean Concept Coverage : "
    f"{mean_concept_coverage:.1f}%"
)

print(
    f"Unique Top-5 Avg      : "
    f"{mean_unique_top5:.1f} / "
    f"{OPTIMIZED_TOP_K}"
)

print("=" * 90)


# ============================================================
# INSPECT TOP-K PER QUERY
# ============================================================

for test_id, results in (
    retrieval_outputs.items()
):

    print(
        f"\n{'=' * 90}"
    )

    print(
        f"{test_id} — "
        f"TOP-{OPTIMIZED_TOP_K} RETRIEVED DOCUMENTS"
    )

    print(
        "=" * 90
    )


    if results.empty:

        print(
            "No documents retrieved."
        )

        continue


    display(
        results[
            [
                "rank",
                "source_family",
                "collection",
                "identifier",
                "title",
                "shard_name",
                "matched_terms",
                "bm25_score"
            ]
        ].round({
            "bm25_score": 4
        })
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

expected_result_count = (
    OPTIMIZED_TOP_K
)


all_queries_returned_top_k = (

    RETRIEVAL_BENCHMARK[
        "results"
    ].eq(
        expected_result_count
    ).all()
)


all_top_k_unique = (

    RETRIEVAL_BENCHMARK[
        "unique_top5"
    ].eq(
        expected_result_count
    ).all()
)


print(
    "\n" + "=" * 90
)

print(
    "RETRIEVAL VALIDATION"
)

print("=" * 90)


print(
    f"All Queries Returned Top-{OPTIMIZED_TOP_K} : "
    f"{all_queries_returned_top_k}"
)

print(
    f"All Top-{OPTIMIZED_TOP_K} Results Unique   : "
    f"{all_top_k_unique}"
)

print("=" * 90)


if not all_queries_returned_top_k:

    raise RuntimeError(
        "One or more retrieval tests did not "
        f"return Top-{OPTIMIZED_TOP_K} results."
    )


if not all_top_k_unique:

    raise RuntimeError(
        "Duplicate documents detected within "
        "one or more retrieval Top-K results."
    )


print(
    "✓ Optimized retrieval completed "
    "successfully across all benchmark queries."
)

RETRIEVAL QUALITY + LATENCY BENCHMARK

Running Q1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

,rank,collection,identifier,title,matched_terms,bm25_score
0,1,3GPP-TSG,S2-2208906,S2-2208906,3,11.5782
1,2,3GPP-TSG,C4-231563,C4-231563,3,11.5260
2,3,3GPP-TSG,C4-231565,C4-231565,3,11.5163
3,4,3GPP-TSG,C4-193074,C4-193074,3,11.5139
4,5,3GPP-TSG,C4-232098,C4-232098,3,11.4719



Q3 — TOP-5 RETRIEVED INPUTS


,rank,collection,identifier,title,matched_terms,bm25_score
0,1,3GPP-TSG,R3-255169,R3-255169,3,7.5281
1,2,3GPP-TSG,R3-171024,R3-171024,3,7.5114
2,3,3GPP-TSG,R3-186758,R3-186758,3,7.4245
3,4,3GPP-TSG,S2-1905284,S2-1905284,3,7.4037
4,5,3GPP-TSG,R3-240993,R3-240993,3,7.3617



Q4 — TOP-5 RETRIEVED INPUTS


,rank,collection,identifier,title,matched_terms,bm25_score
0,1,IETF-Drafts,draft-kazuho-quic-quic-on-streams,QUIC on Streams,3,5.8220
1,2,IETF-Drafts,draft-shade-quic-http2-mapping,HTTP/2 Semantics Using The QUIC Transport Prot...,3,5.7729
2,3,IETF-Drafts,draft-ietf-quic-qmux-1,QUIC ...,3,5.7708
3,4,IETF-Drafts,draft-llg-opsawg-ipfix-over-quic-2,OPSAWG ...,3,5.7672
4,5,IETF-Drafts,draft-opik-quic-qmux-1,QUIC ...,3,5.7055


**Observation — Build-Time Retrieval Quality + Latency Evaluation**

The build-time tests confirmed useful retrieval across 5G Core, mobility and IETF domains while preserving document provenance and unique Top-K results. They also exposed an important lexical-search limitation: **high BM25 relevance is not the same as normative authority**.

***Key Decision:*** Retain source-family, collection, document-type and identifier metadata for downstream authority/grounding analysis. Formal Version A/B comparison is performed only with the frozen runtime Benchmark v2 artifacts.


# **SECTION 4 — Independent Persistence Validation**

## **Cell 16 — Independent Kaggle Reload Validation**

In [ ]:
# ============================================================
# CELL 16 — INDEPENDENT KAGGLE RELOAD VALIDATION
# ============================================================

import os
import shutil
import time
import subprocess

import duckdb
import pandas as pd

from google.colab import userdata


# ============================================================
# VALIDATION CONFIGURATION
# ============================================================

# Set False when independent Kaggle validation
# is not required for the current run.
VALIDATE_KAGGLE_RELOAD = True


KAGGLE_DATASET = (
    "cliffordimaguezegie/"
    "telecom-bm25-indexed-knowledge-base"
)


# Persisted build manifest stored with the Kaggle artifact.
PERSISTED_MANIFEST_FILE = (
    "bm25_shard_manifest.csv"
)


# Smallest shard selected for independent validation.
TEST_SHARD_NAME = (
    "tcc_ietf_proceedings_00"
)

TEST_FILE = (
    f"{TEST_SHARD_NAME}.duckdb"
)


# Frozen Version B artifact expectations.
EXPECTED_PERSISTED_SHARDS = 21
EXPECTED_PERSISTED_RECORDS = 1_780_938


# Independent reload location.
RELOAD_DIR = (
    WORK_DIR
    / "kaggle_reload_validation"
)


# ============================================================
# OPTIONAL VALIDATION PATH
# ============================================================

if not VALIDATE_KAGGLE_RELOAD:

    print("=" * 80)
    print("INDEPENDENT KAGGLE RELOAD VALIDATION")
    print("=" * 80)

    print(
        "Validation Enabled : False"
    )

    print(
        "✓ Kaggle reload validation skipped."
    )

    print("=" * 80)


else:

    # ========================================================
    # KAGGLE AUTHENTICATION — COLAB SECRET
    # ========================================================

    try:

        kaggle_api_token = (
            userdata.get(
                "KAGGLE_API_TOKEN"
            )
        )

    except Exception as exc:

        raise RuntimeError(
            "Unable to read KAGGLE_API_TOKEN "
            "from Google Colab Secrets."
        ) from exc


    if not kaggle_api_token:

        raise RuntimeError(
            "KAGGLE_API_TOKEN was not found "
            "in Google Colab Secrets."
        )


    os.environ[
        "KAGGLE_API_TOKEN"
    ] = kaggle_api_token


    print("=" * 80)
    print("INDEPENDENT KAGGLE RELOAD VALIDATION")
    print("=" * 80)

    print(
        "Kaggle Authentication : ✓ CONFIGURED"
    )

    print(
        f"Dataset               : "
        f"{KAGGLE_DATASET}"
    )

    print(
        f"Validation Shard      : "
        f"{TEST_SHARD_NAME}"
    )

    print("=" * 80)


    # ========================================================
    # VERIFY KAGGLE CLI
    # ========================================================

    version_check = subprocess.run(
        [
            "kaggle",
            "--version"
        ],
        capture_output=True,
        text=True
    )


    if version_check.returncode != 0:

        raise RuntimeError(
            "Kaggle CLI is unavailable. "
            "Check the Cell 0 installation."
        )


    print(
        f"✓ {version_check.stdout.strip()}"
    )


    # ========================================================
    # VERIFY DATASET ACCESS
    # ========================================================

    dataset_check = subprocess.run(
        [
            "kaggle",
            "datasets",
            "files",
            KAGGLE_DATASET,
            "--page-size",
            "50"
        ],
        capture_output=True,
        text=True
    )


    if dataset_check.returncode != 0:

        raise RuntimeError(
            "Kaggle dataset access failed.\n\n"
            f"{dataset_check.stderr}"
        )


    print(
        "✓ Kaggle dataset access validated."
    )


    # ========================================================
    # PREPARE CLEAN INDEPENDENT RELOAD DIRECTORY
    # ========================================================

    if RELOAD_DIR.exists():

        shutil.rmtree(
            RELOAD_DIR
        )


    RELOAD_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


    # ========================================================
    # DOWNLOAD PERSISTED MANIFEST FROM KAGGLE
    # ========================================================

    print(
        "\nDownloading persisted build manifest..."
    )


    manifest_download_start = (
        time.perf_counter()
    )


    manifest_download = subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            KAGGLE_DATASET,

            "-f",
            PERSISTED_MANIFEST_FILE,

            "-p",
            str(RELOAD_DIR),

            "--unzip"
        ],
        capture_output=True,
        text=True
    )


    manifest_download_elapsed = (
        time.perf_counter()
        - manifest_download_start
    )


    if manifest_download.returncode != 0:

        raise RuntimeError(
            "Kaggle manifest download failed.\n\n"
            f"{manifest_download.stderr}"
        )


    RELOADED_MANIFEST = (
        RELOAD_DIR
        / PERSISTED_MANIFEST_FILE
    )


    if not RELOADED_MANIFEST.exists():

        raise FileNotFoundError(
            "Downloaded Kaggle manifest was "
            f"not found: {RELOADED_MANIFEST}"
        )


    print(
        f"✓ Manifest downloaded in "
        f"{manifest_download_elapsed:.2f} sec"
    )


    # ========================================================
    # VALIDATE PERSISTED MANIFEST
    # ========================================================

    persisted_manifest = (
        pd.read_csv(
            RELOADED_MANIFEST
        )
    )


    required_manifest_columns = {
        "shard_name",
        "source_family",
        "collection",
        "records"
    }


    missing_columns = (
        required_manifest_columns
        - set(
            persisted_manifest.columns
        )
    )


    if missing_columns:

        raise RuntimeError(
            "Persisted Kaggle manifest is "
            "missing required columns: "
            f"{sorted(missing_columns)}"
        )


    persisted_shard_count = len(
        persisted_manifest
    )


    persisted_record_count = int(
        persisted_manifest[
            "records"
        ].sum()
    )


    if (
        persisted_shard_count
        != EXPECTED_PERSISTED_SHARDS
    ):

        raise RuntimeError(
            "Persisted Kaggle manifest shard "
            "count does not match the frozen "
            "Version B architecture: "
            f"{persisted_shard_count} != "
            f"{EXPECTED_PERSISTED_SHARDS}"
        )


    if (
        persisted_record_count
        != EXPECTED_PERSISTED_RECORDS
    ):

        raise RuntimeError(
            "Persisted Kaggle manifest record "
            "count does not match the frozen "
            "Version B corpus: "
            f"{persisted_record_count:,} != "
            f"{EXPECTED_PERSISTED_RECORDS:,}"
        )


    print(
        "✓ Persisted Kaggle manifest validated."
    )

    print(
        f"  Shards  : "
        f"{persisted_shard_count}"
    )

    print(
        f"  Records : "
        f"{persisted_record_count:,}"
    )


    # ========================================================
    # RESOLVE EXPECTED TEST-SHARD STATE
    # FROM PERSISTED MANIFEST
    # ========================================================

    test_manifest_rows = (
        persisted_manifest.loc[
            persisted_manifest[
                "shard_name"
            ] == TEST_SHARD_NAME
        ]
    )


    if len(test_manifest_rows) != 1:

        raise RuntimeError(
            "Unable to resolve the validation "
            "shard from the persisted Kaggle manifest."
        )


    EXPECTED_TEST_RECORDS = int(
        test_manifest_rows.iloc[
            0
        ][
            "records"
        ]
    )


    print(
        f"  Test Shard Records : "
        f"{EXPECTED_TEST_RECORDS:,}"
    )


    # ========================================================
    # DOWNLOAD TEST SHARD FRESH FROM KAGGLE
    # ========================================================

    print(
        "\nDownloading independent test shard..."
    )


    download_start = (
        time.perf_counter()
    )


    download_result = subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            KAGGLE_DATASET,

            "-f",
            TEST_FILE,

            "-p",
            str(RELOAD_DIR),

            "--unzip"
        ],
        capture_output=True,
        text=True
    )


    download_elapsed = (
        time.perf_counter()
        - download_start
    )


    if download_result.returncode != 0:

        raise RuntimeError(
            "Kaggle shard download failed.\n\n"
            f"{download_result.stderr}"
        )


    RELOADED_SHARD = (
        RELOAD_DIR
        / TEST_FILE
    )


    if not RELOADED_SHARD.exists():

        raise FileNotFoundError(
            "Downloaded Kaggle shard was not "
            f"found: {RELOADED_SHARD}"
        )


    print(
        f"✓ Test shard downloaded in "
        f"{download_elapsed:.2f} sec"
    )


    # ========================================================
    # OPEN RELOADED SHARD READ-ONLY
    # ========================================================

    conn = duckdb.connect(
        str(RELOADED_SHARD),
        read_only=True
    )


    try:

        conn.execute(
            "LOAD fts"
        )

    except Exception:

        conn.close()

        raise RuntimeError(
            "DuckDB FTS extension could not "
            "be loaded for the reloaded shard."
        )


    # ========================================================
    # TABLE VALIDATION
    # ========================================================

    table_ready = (
        conn.execute(
            """
            SELECT COUNT(*)

            FROM information_schema.tables

            WHERE table_name =
                  'telecom_shard'
            """
        ).fetchone()[0]
        > 0
    )


    if not table_ready:

        conn.close()

        raise RuntimeError(
            "Reloaded shard does not contain "
            "the telecom_shard table."
        )


    record_count = int(
        conn.execute(
            """
            SELECT COUNT(*)
            FROM telecom_shard
            """
        ).fetchone()[0]
    )


    # ========================================================
    # FTS / BM25 VALIDATION
    # ========================================================

    fts_ready = (
        conn.execute(
            """
            SELECT COUNT(*)

            FROM information_schema.schemata

            WHERE schema_name =
                  'fts_main_telecom_shard'
            """
        ).fetchone()[0]
        > 0
    )


    if not fts_ready:

        conn.close()

        raise RuntimeError(
            "Reloaded shard does not contain "
            "the persistent FTS/BM25 schema."
        )


    index_stats = (
        conn.execute(
            """
            SELECT
                num_docs,
                avgdl

            FROM
                fts_main_telecom_shard.stats
            """
        ).df()
    )


    if index_stats.empty:

        conn.close()

        raise RuntimeError(
            "FTS statistics table returned "
            "no metadata."
        )


    fts_num_docs = int(
        index_stats.iloc[
            0
        ][
            "num_docs"
        ]
    )


    # ========================================================
    # SAMPLE METADATA READ
    # ========================================================

    sample_docs = (
        conn.execute(
            """
            SELECT
                identifier,
                title,
                collection

            FROM telecom_shard

            LIMIT 5
            """
        ).df()
    )


    conn.close()


    # ========================================================
    # FINAL VALIDATION
    # ========================================================

    reload_valid = (

        table_ready

        and fts_ready

        and record_count
            == EXPECTED_TEST_RECORDS

        and fts_num_docs
            == EXPECTED_TEST_RECORDS

        and not sample_docs.empty
    )


    downloaded_size_gb = (
        RELOADED_SHARD.stat().st_size
        / (1024 ** 3)
    )


    # ========================================================
    # REPORT
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "KAGGLE RELOAD VALIDATION RESULT"
    )

    print("=" * 80)


    print(
        f"Persisted Shards    : "
        f"{persisted_shard_count}"
    )

    print(
        f"Persisted Records   : "
        f"{persisted_record_count:,}"
    )

    print(
        f"Validation Shard    : "
        f"{TEST_SHARD_NAME}"
    )

    print(
        f"Downloaded Size     : "
        f"{downloaded_size_gb:.2f} GB"
    )

    print(
        f"Download Time       : "
        f"{download_elapsed:.2f} sec"
    )

    print(
        f"Expected Records    : "
        f"{EXPECTED_TEST_RECORDS:,}"
    )

    print(
        f"Table Records       : "
        f"{record_count:,}"
    )

    print(
        f"FTS Documents       : "
        f"{fts_num_docs:,}"
    )

    print(
        f"FTS Index Ready     : "
        f"{fts_ready}"
    )

    print(
        f"Metadata Read Ready : "
        f"{not sample_docs.empty}"
    )

    print("=" * 80)


    print(
        "\nIndex Statistics:"
    )

    display(
        index_stats
    )


    print(
        "\nSample Reloaded Documents:"
    )

    display(
        sample_docs
    )


    # ========================================================
    # STRICT RESULT
    # ========================================================

    if not reload_valid:

        raise RuntimeError(
            "Independent Kaggle artifact "
            "reload validation failed."
        )


    print(
        "\n✓ INDEPENDENT KAGGLE "
        "RELOAD VALIDATION PASSED"
    )

    print(
        "✓ Persisted Kaggle manifest "
        "validated independently."
    )

    print(
        "✓ Fresh DuckDB shard successfully "
        "downloaded from Kaggle."
    )

    print(
        "✓ Persistent telecom_shard "
        "table retained."
    )

    print(
        "✓ Persistent FTS/BM25 "
        "index retained."
    )

    print(
        "✓ Table record count and FTS "
        "document count match the "
        "persisted Kaggle manifest."
    )

**Observation — Independent Persistence Validation**

Cell 16 is retained as an optional reproducibility utility that downloads a persisted shard into a fresh location and verifies its DuckDB table, record count and FTS/BM25 index against the Kaggle manifest.

The previously attached output did not correspond to the current Cell 16 source and has been removed rather than presented as fresh validation evidence. The Version B runtime notebooks separately demonstrate restoration and validation of the complete 21-shard artifact.

***Key Decision:*** Do not claim a fresh independent reload unless the visible output was produced by this exact cell in the current run.
